# 03 — Feature Selection (Boruta)

**Pipeline position:** Step 3 of 7

**Purpose:** Apply the Boruta algorithm on the training set only (preventing
data leakage) to identify confirmed features.  The selected feature list is
then applied uniformly to all three dataset splits.

**License:** MIT

## Prerequisites
- Run `02_data_splitting.ipynb` first
- Inputs: `./Data/data_training_imputed.csv`, `./Data/data_internal_validation_imputed.csv`, `./Data/data_temporal_validation_imputed.csv`

## Outputs
- `selected_features.csv`
- `boruta_results.csv`
- `data_training_selected.csv`
- `data_internal_validation_selected.csv`
- `data_temporal_validation_selected.csv`


In [ ]:
# SPDX-License-Identifier: MIT
# ============================================================
# Dependency check — run this cell first
# ============================================================
import importlib.util

_required = ['pandas', 'numpy', 'sklearn', 'boruta', 'matplotlib']
_missing = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    raise ImportError(
        f"Missing packages: {_missing}\n"
        f"Install with: pip install {' '.join(_missing)}"
    )
print("Core dependencies satisfied.")

In [ ]:
# ============================================================
# Configuration
# ============================================================
RANDOM_STATE   = 42
OUTCOME_VAR    = "SpontAbortion"
N_ESTIMATORS   = 500        # number of trees in Boruta's RF
MAX_ITER       = 100        # maximum Boruta iterations
PERC           = 100        # shadow feature percentile threshold
ALPHA          = 0.05       # significance level for binomial test
DATA_DIR       = "./Data"


In [21]:
"""
Boruta feature selection script (training set only) + automatic result plot
==========================================================================
Principle:
  - Feature selection exclusively on training set (no data leakage)
  - Extract importance score for each feature
  - Apply selected feature list uniformly to all three splits
  - Automatically generate Boruta result plot

Dependencies:
  pip install pandas numpy scikit-learn boruta matplotlib
"""

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from boruta import BorutaPy
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import time

warnings.filterwarnings("ignore")

# ============================================================
# 1. Configuration
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VAR = "BaseInfoDate"

# Input files
TRAIN_FILE = "./Data/data_training_imputed.csv"
INTVAL_FILE = "./Data/data_internal_validation_imputed.csv"
TEMPVAL_FILE = "./Data/data_temporal_validation_imputed.csv"

# Boruta parameters
BORUTA_CONFIG = {
    "n_estimators": 100,
    "max_depth": 7,
    "max_iter": 100,
    "perc": 100,
    "alpha": 0.05,
    "random_state": 42,
    "n_jobs": -1,
}

# Large-sample sub-sampling
USE_SUBSAMPLE = True
SUBSAMPLE_SIZE = 500000

# Plot parameters
PLOT_CONFIG = {
    "top_n_confirmed": 15,     # number of confirmed features shown individually in bar chart
    "top_n_rejected": 5,       # number of rejected features shown individually in bar chart
    "dpi": 300,
    "color_confirmed": "#5BB37A",
    "color_rejected": "#E8637A",
    "color_text": "#333333",
    "color_pie_label_bg": "#D0E8F0",
}

# Variable type lists
CONTINUOUS_VARS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
    "Hemoglobin", "RBC", "Platelet", "WBC",
    "NeutrophilPct", "LymphocytePct",
    "Glucose", "WifeALT", "WifeCreat", "TSH",
    "HusbALT", "HusbCreat",
    "LeftTestVol", "RightTestVol",
]

ORDINAL_VARS = ["WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc"]

NOMINAL_VARS = [
    "WifeEthnic", "HusbandEthnic",
    "WifeResident", "HusbandResident",
    "WifeABO", "HusbABO",
    "CycleRegular", "MenstrualVol", "Dysmenorrhea",
    "WifeMental", "WifeIntel", "WifeFace", "WifePosture",
    "WifeFaceSpec", "WifeSkinHair",
    "HusbMental", "HusbIntel", "HusbFace", "HusbPosture",
    "HusbFaceSpec", "HusbSkinHair",
    "WifePubHair", "Breast", "Vulva",
    "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
    "Testis", "Epididymis", "VasDeferens", "Varicocele",
    "WifeUrine", "HusbUrine",
    "WifeHepB", "HusbHepB",
]

BINARY_VARS = [
    "WifeDiseaseHx", "WifeBirthDefect", "GynDisease",
    "WifeMedication", "WifeVaccine", "Contraception",
    "PrevPregnancy", "Consanguinity", "WifeFamConsang", "WifeFamDisease",
    "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
    "WifeSmoke", "WifePassSmoke", "WifeAlcohol",
    "WifeDrugUse", "WifeHalitosis", "WifeGumBleed",
    "WifeEnvExpose", "WifeStress", "WifeRelatStress",
    "WifeFinance", "WifeReady",
    "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
    "WifeLiverSpleen", "WifeExtremity", "WifeGynExam",
    "WifeRh", "HusbRh",
    "RubellaIgG", "CMVIgG", "CMVIgM",
    "ToxoIgG", "ToxoIgM",
    "SyphilisScr", "HusbSyphilis",
    "GynUltrasound",
    "HusbDiseaseHis", "HusbBirthDefect", "HusbAndrologyDis",
    "HusbMedication", "HusbVaccinated",
    "HusbFamConsang", "HusbFamDisease",
    "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
    "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse",
    "HusbEnvExpose", "HusbStress", "HusbRelatStress",
    "HusbFinance", "HusbReady",
    "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
    "HusbLiverSpleen", "HusbExtremity", "HusbAndroExam",
]

# ============================================================
# 2. Load the three dataset splits
# ============================================================
print("=" * 70)
print("STEP 1: Load imputed datasets")
print("=" * 70)

df_train = pd.read_csv(TRAIN_FILE)
df_intval = pd.read_csv(INTVAL_FILE)
df_tempval = pd.read_csv(TEMPVAL_FILE)

print(f"  Training set          : {len(df_train):,}")
print(f"  Internal val  : {len(df_intval):,}")
print(f"  Temporal val  : {len(df_tempval):,}")

# ============================================================
# 3. Encode training set for Boruta
# ============================================================
print(f"\n{'='*70}")
print("STEP 2: Encode training set")
print("=" * 70)

drop_cols = [INDEX_VAR, DATE_VAR, "Dataset"]
df_train_model = df_train.drop(
    columns=[c for c in drop_cols if c in df_train.columns], errors="ignore"
)

if df_train_model[OUTCOME_VAR].dtype == object or df_train_model[OUTCOME_VAR].dtype == bool:
    df_train_model[OUTCOME_VAR] = df_train_model[OUTCOME_VAR].map({
        True: 1, False: 0, "True": 1, "False": 0,
        "TRUE": 1, "FALSE": 0
    })

label_encoders = {}
for col in ORDINAL_VARS + NOMINAL_VARS:
    if col in df_train_model.columns and df_train_model[col].dtype == object:
        le = LabelEncoder()
        df_train_model[col] = le.fit_transform(df_train_model[col].astype(str))
        label_encoders[col] = le

for col in BINARY_VARS:
    if col in df_train_model.columns and df_train_model[col].dtype == object:
        df_train_model[col] = df_train_model[col].map({
            True: 1, False: 0, "True": 1, "False": 0,
            "TRUE": 1, "FALSE": 0
        })

df_train_model = df_train_model.apply(pd.to_numeric, errors="coerce")

n_miss = df_train_model.isnull().sum().sum()
if n_miss > 0:
    print(f"  Warning: {n_miss} missing values remain in training set — dropping rows")
    df_train_model = df_train_model.dropna()

y_train = df_train_model[OUTCOME_VAR].astype(int).values
X_train = df_train_model.drop(columns=[OUTCOME_VAR])
feature_names = X_train.columns.tolist()

print(f"  Features: {len(feature_names)}")
print(f"  Training samples: {len(y_train):,}")
print(f"  Outcome distribution: SA={y_train.sum():,} ({y_train.mean()*100:.2f}%)")

# ============================================================
# 4. Stratified subsampling (training set only)
# ============================================================
if USE_SUBSAMPLE and len(y_train) > SUBSAMPLE_SIZE:
    print(f"\n  Stratified sub-sampling: {len(y_train):,} → {SUBSAMPLE_SIZE:,}")
    rng = np.random.RandomState(42)
    pos_idx = np.where(y_train == 1)[0]
    neg_idx = np.where(y_train == 0)[0]
    pos_ratio = len(pos_idx) / len(y_train)

    n_pos = int(SUBSAMPLE_SIZE * pos_ratio)
    n_neg = SUBSAMPLE_SIZE - n_pos

    sampled_pos = rng.choice(pos_idx, size=min(n_pos, len(pos_idx)), replace=False)
    sampled_neg = rng.choice(neg_idx, size=min(n_neg, len(neg_idx)), replace=False)
    sampled_idx = np.concatenate([sampled_pos, sampled_neg])
    rng.shuffle(sampled_idx)

    X_boruta = X_train.iloc[sampled_idx].values
    y_boruta = y_train[sampled_idx]
    print(f"  After sub-sampling: SA={y_boruta.sum():,} ({y_boruta.mean()*100:.2f}%)")
else:
    X_boruta = X_train.values
    y_boruta = y_train

# ============================================================
# 5. Boruta feature selection (training set only)
# ============================================================
print(f"\n{'='*70}")
print("STEP 3: Boruta feature selection (training set only, no data leakage)")
print("=" * 70)
print(f"  RF: {BORUTA_CONFIG['n_estimators']} trees, "
      f"max_depth {BORUTA_CONFIG['max_depth']}")
print(f"  Max iterations: {BORUTA_CONFIG['max_iter']}")
print(f"  Significance level: α = {BORUTA_CONFIG['alpha']}")

rf = RandomForestClassifier(
    n_estimators=BORUTA_CONFIG["n_estimators"],
    max_depth=BORUTA_CONFIG["max_depth"],
    class_weight="balanced",
    random_state=BORUTA_CONFIG["random_state"],
    n_jobs=BORUTA_CONFIG["n_jobs"],
)

boruta_selector = BorutaPy(
    estimator=rf,
    n_estimators="auto",
    perc=BORUTA_CONFIG["perc"],
    alpha=BORUTA_CONFIG["alpha"],
    max_iter=BORUTA_CONFIG["max_iter"],
    random_state=BORUTA_CONFIG["random_state"],
    verbose=2,
)

t_start = time.time()
boruta_selector.fit(X_boruta, y_boruta)
t_elapsed = time.time() - t_start
print(f"\nBoruta complete. Elapsed: {t_elapsed/60:.1f} min")

# ============================================================
# 6. Extract feature importance scores (consistent with R Boruta package)
# ============================================================
print(f"\n{'='*70}")
print("STEP 4: Extract feature importance scores (R-Boruta compatible logic)")
print("=" * 70)

# ---------------------------------------------------------------
# R Boruta package importance score logic:
#
#   1. Each iteration:
#      - Create a shadow feature for each real feature (shuffle columns)
#      - Train RF with [real features + shadow features] combined
#      - Extract each real feature's importance value for this iteration
#
#   2. Aggregate across iterations:
#      - R's attStats() outputs meanImp / medianImp / minImp / maxImp
#      - R plots typically use meanImp or medianImp
#      - normHits = fraction of iterations where feature beats MZSA
#
#   3. Statistical test:
#      - Each iteration: compare real feature vs shadow max (MZSA)
#      - Binomial test applied across iterations
#
# Python BorutaPy executes the same logic internally but does not
# expose intermediate results. This code replicates R's approach.
# ---------------------------------------------------------------

N_IMPORTANCE_RUNS = BORUTA_CONFIG["max_iter"]  # same iteration count as Boruta run
n_features = X_boruta.shape[1]

print(f"  Replicating R-Boruta logic: {N_IMPORTANCE_RUNS} importance extraction iterations...")
print(f"  Each iteration: real features ({n_features}) + shadow features ({n_features}) → RF")

importance_history = np.zeros((N_IMPORTANCE_RUNS, n_features))  # real features
shadow_max_history = np.zeros(N_IMPORTANCE_RUNS)                # shadow max per iteration

rng_imp = np.random.RandomState(BORUTA_CONFIG["random_state"])

for run in range(N_IMPORTANCE_RUNS):
    # Create shadow features: independently shuffle each column
    X_shadow = X_boruta.copy()
    for col_i in range(n_features):
        rng_imp.shuffle(X_shadow[:, col_i])

    # Concatenate real and shadow features
    X_combined = np.hstack([X_boruta, X_shadow])

    # Train Random Forest
    rf_run = RandomForestClassifier(
        n_estimators=BORUTA_CONFIG["n_estimators"],
        max_depth=BORUTA_CONFIG["max_depth"],
        class_weight="balanced",
        random_state=BORUTA_CONFIG["random_state"] + run,
        n_jobs=BORUTA_CONFIG["n_jobs"],
    )
    rf_run.fit(X_combined, y_boruta)

    # Extract feature importances
    all_importances = rf_run.feature_importances_
    importance_history[run, :] = all_importances[:n_features]        # real features
    shadow_max_history[run] = all_importances[n_features:].max()     # shadow max value

    if (run + 1) % 20 == 0 or run == 0:
        print(f"    Iter {run+1:3d}/{N_IMPORTANCE_RUNS} | "
              f"Shadow max (MZSA): {shadow_max_history[run]:.6f}")

    del rf_run, X_shadow, X_combined

# Aggregate statistics (equivalent to R attStats())
importance_mean = importance_history.mean(axis=0)     # R: meanImp
importance_median = np.median(importance_history, axis=0)  # R: medianImp
importance_min = importance_history.min(axis=0)       # R: minImp
importance_max = importance_history.max(axis=0)       # R: maxImp
importance_std = importance_history.std(axis=0)       # standard deviation

# normHits: fraction of iterations where feature importance exceeds MZSA
hits = np.zeros(n_features)
for run in range(N_IMPORTANCE_RUNS):
    hits += (importance_history[run, :] > shadow_max_history[run]).astype(float)
norm_hits = hits / N_IMPORTANCE_RUNS                  # R: normHits

# Use meanImp as the plotting importance score (consistent with R default)
importance_scores = importance_mean

print(f"\n  Importance score statistics (across {N_IMPORTANCE_RUNS} iterations):")
print(f"    meanImp   range: [{importance_mean.min():.6f}, {importance_mean.max():.6f}]")
print(f"    medianImp range: [{importance_median.min():.6f}, {importance_median.max():.6f}]")
print(f"    MZSA mean: {shadow_max_history.mean():.6f}")
print(f"    normHits range: [{norm_hits.min():.4f}, {norm_hits.max():.4f}]")

# ============================================================
# 7. Organise results
# ============================================================
def get_var_type(var):
    if var in CONTINUOUS_VARS:
        return "Continuous"
    elif var in ORDINAL_VARS:
        return "Ordinal"
    elif var in NOMINAL_VARS:
        return "Nominal"
    elif var in BINARY_VARS:
        return "Binary"
    return "Other"

results = pd.DataFrame({
    "Variable": feature_names,
    "Confirmed": boruta_selector.support_,
    "Tentative": boruta_selector.support_weak_,
    "Rank": boruta_selector.ranking_,
    "meanImp": importance_mean,
    "medianImp": importance_median,
    "minImp": importance_min,
    "maxImp": importance_max,
    "stdImp": importance_std,
    "normHits": norm_hits,
    "ImportanceScore": importance_scores,    # = meanImp, used for plotting
    "VariableType": [get_var_type(f) for f in feature_names],
})
results = results.sort_values("ImportanceScore", ascending=False)

confirmed = results[results["Confirmed"]].copy()
tentative = results[results["Tentative"]].copy()
rejected = results[~results["Confirmed"] & ~results["Tentative"]].copy()

print(f"\n{'='*70}")
print("Boruta results summary")
print(f"{'='*70}")
print(f"  Total features  : {len(feature_names)}")
print(f"  Confirmed       : {len(confirmed)}")
print(f"  Tentative       : {len(tentative)}")
print(f"  Rejected        : {len(rejected)}")

print(f"\n--- Confirmed features ({len(confirmed)}) ---")
for _, row in confirmed.iterrows():
    print(f"  {row['Feature']:30s} | {row['VariableType']:8s} | "
          f"rank {row['Rank']:2.0f} | importance {row['ImportanceScore']:.4f}")

if len(tentative) > 0:
    print(f"\n--- Tentative features ({len(tentative)}) ---")
    for _, row in tentative.iterrows():
        print(f"  {row['Feature']:30s} | {row['VariableType']:8s} | "
              f"rank {row['Rank']:2.0f} | importance {row['ImportanceScore']:.4f}")

# Count by variable type
print(f"\n--- Selected by variable type ---")
for vtype in ["Continuous", "Ordinal", "Nominal", "Binary"]:
    total = len(results[results["VariableType"] == vtype])
    selected = len(confirmed[confirmed["VariableType"] == vtype])
    print(f"  {vtype:8s}: {selected:3d} / {total} selected")

# Save full results (including importance scores)
results.to_csv("boruta_results.csv", index=False, encoding="utf-8-sig")
print(f"\n  Full results (with importance scores) saved to: boruta_results.csv")

# ============================================================
# 8. Plot Boruta feature selection results
# ============================================================
print(f"\n{'='*70}")
print("STEP 5: Plotting Boruta feature selection results")
print("=" * 70)

PC = PLOT_CONFIG
TOP_N_C = PC["top_n_confirmed"]
TOP_N_R = PC["top_n_rejected"]

# Sort
confirmed_sorted = confirmed.sort_values("ImportanceScore", ascending=False)
rejected_sorted = rejected.sort_values("ImportanceScore", ascending=False)

# Top confirmed
top_confirmed = confirmed_sorted.head(TOP_N_C)
other_confirmed_sum = confirmed_sorted.iloc[TOP_N_C:]["ImportanceScore"].sum()
n_other_confirmed = len(confirmed_sorted) - TOP_N_C

# Top rejected
top_rejected = rejected_sorted.head(TOP_N_R)
other_rejected_sum = rejected_sorted.iloc[TOP_N_R:]["ImportanceScore"].sum()
if len(tentative) > 0:
    other_rejected_sum += tentative["ImportanceScore"].sum()
n_other_rejected = len(rejected_sorted) - TOP_N_R + len(tentative)

# Assemble plot data (bottom to top: rejected summary → rejected → confirmed summary → confirmed)
labels = []
values = []
colors = []

# Rejected summary bar (bottom)
labels.append(f"Sum of Other Rejected Feature")
values.append(round(other_rejected_sum, 3))
colors.append(PC["color_rejected"])

# Top rejected (low to high)
for _, row in top_rejected.iloc[::-1].iterrows():
    labels.append(row["Variable"])
    values.append(round(row["ImportanceScore"], 3))
    colors.append(PC["color_rejected"])

# Confirmed summary bar
labels.append(f"Sum of Other Confirmed Feature")
values.append(round(other_confirmed_sum, 3))
colors.append(PC["color_confirmed"])

# Top confirmed (low to high)
for _, row in top_confirmed.iloc[::-1].iterrows():
    labels.append(row["Variable"])
    values.append(round(row["ImportanceScore"], 3))
    colors.append(PC["color_confirmed"])


# ============================================================
# 9. Apply selected features to all three dataset splits
# ============================================================
print(f"\n{'='*70}")
print("STEP 6: Applying training-set selected features to all splits")
print("=" * 70)

selected_features = confirmed["Variable"].tolist()
print(f"  Selected features: {len(selected_features)}")

keep_cols_base = [OUTCOME_VAR] + selected_features
keep_cols_with_meta = (
    [c for c in [INDEX_VAR, DATE_VAR] if c in df_train.columns] + keep_cols_base
)

datasets = {
    "Training": (df_train, "data_training_selected.csv"),
    "Internal Validation": (df_intval, "data_internal_validation_selected.csv"),
    "Temporal Validation": (df_tempval, "data_temporal_validation_selected.csv"),
}

for name, (df_full, out_path) in datasets.items():
    available_cols = [c for c in keep_cols_with_meta if c in df_full.columns]
    df_selected = df_full[available_cols].copy()
    df_selected.to_csv(out_path, index=False)

    if OUTCOME_VAR in df_selected.columns:
        y = df_selected[OUTCOME_VAR].map({
            True: 1, False: 0, "True": 1, "False": 0,
            "TRUE": 1, "FALSE": 0, 1: 1, 0: 0
        }).astype(int)
        print(f"  {name:12s}: N={len(df_selected):>7,} | "
              f"features={len(selected_features)} | "
              f"SA={y.sum():>5,} ({y.mean()*100:.2f}%) | "
              f"→ {out_path}")

pd.DataFrame({"selected_features": selected_features}).to_csv(
    "selected_features.csv", index=False
)

# ============================================================
# 10. Summary
# ============================================================
print(f"\n{'='*70}")
print("Summary")
print("=" * 70)
print(f"""
  Complete pipeline (data-leakage-free):
  ┌─ Raw data (N ≈ 402,226)
  ├─ Step 1: Split → Training / Internal Validation / Temporal Validation
  ├─ Step 2: Independent imputation per split
  ├─ Step 3: Boruta executed on training set only
  ├─ Step 4: Extract feature importance scores (feature_importances_)
  ├─ Step 5: Plot results
  ├─ Step 6: Apply selected features uniformly to all splits
  └─ Next: Train models on training set → evaluate on validation sets

  Output files:
    boruta_results.csv                         all feature selection results (meanImp / medianImp / normHits)
    selected_features.csv                      selected feature list
    boruta_feature_selection_plot.tiff          bar chart (TIFF, 300 DPI)
    boruta_feature_selection_plot.png           bar chart (PNG)
    boruta_importance_boxplot.tiff              box plot (TIFF, 300 DPI)
    boruta_importance_boxplot.png               box plot (PNG)
    data_training_selected.csv                 training set (selected features only)
    data_internal_validation_selected.csv      internal validation set (selected features only)
    data_temporal_validation_selected.csv      temporal validation set (selected features only)

  Suggested methods description for manuscript:
    Feature selection was performed using the Boruta algorithm on the
    training set only. In each iteration, shadow features were created
    by randomly permuting each original predictor. A random forest
    classifier (n_estimators = 100, max_depth = 7, class_weight =
    'balanced') was trained on the combined set of original and shadow
    features. The importance of each original feature was compared
    against the maximum importance among all shadow features (MZSA)
    across 100 iterations. Features whose mean importance consistently
    exceeded the MZSA threshold were confirmed as important via a
    binomial test (α = 0.05). Feature importance scores reported are
    the mean importance values (mean decrease in Gini impurity)
    averaged across all iterations. The selected feature set was
    subsequently applied to both the internal and temporal validation
    cohorts for model evaluation.
""")
print("Script completed.")

STEP 1: 读取插补后的数据集
  训练集:       360,363
  内部验证集:   40,041
  时间验证集:   1,822

STEP 2: 编码训练集
  特征数量: 134
  训练集样本: 360,363
  结局分布: SA=11,753 (3.26%)

STEP 3: Boruta 特征选择（仅训练集，无数据泄漏）
  随机森林: 100 棵树, 最大深度 7
  最大迭代: 100
  显著性水平: α = 0.05


KeyboardInterrupt: 

In [19]:

# --- Plot ---
n_bars = len(labels)
fig_height = max(10, n_bars * 0.42)
fig, ax = plt.subplots(figsize=(12, fig_height), facecolor="white")
ax.set_facecolor("white")

y_pos = np.arange(n_bars)
bar_height = 0.7

bars = ax.barh(y_pos, values, height=bar_height, color=colors,
               edgecolor="none", zorder=3)

# Value labels
x_max = max(values) * 1.15
for i, (val, bar) in enumerate(zip(values, bars)):
    ax.text(val + x_max * 0.008, y_pos[i], f"{val:.3f}",
            va="center", ha="left", fontsize=9, fontweight="medium",
            color=PC["color_text"], zorder=4)

# Separator line
sep_y = TOP_N_R + 1 - 0.5
ax.axhline(y=sep_y, color="#CCCCCC", linewidth=0.8, linestyle="--", zorder=2)

# Y axis
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=10, color=PC["color_text"])

ytick_labels = ax.get_yticklabels()
for label in ytick_labels:
    text = label.get_text()
    if "Sum of" in text:
        label.set_fontweight("bold")
        label.set_color(
            PC["color_confirmed"] if "Confirmed" in text else PC["color_rejected"]
        )

# X axis
ax.set_xlim(0, x_max)
ax.set_xlabel("Boruta Feature Importance Score", fontsize=13, fontweight="bold",
              color=PC["color_text"], labelpad=10)
ax.set_ylabel("Features", fontsize=13, fontweight="bold",
              color=PC["color_text"], labelpad=10)

ax.xaxis.grid(True, linestyle="-", alpha=0.15, zorder=1)
ax.set_axisbelow(True)
ax.tick_params(axis="x", labelsize=10, colors=PC["color_text"])
ax.tick_params(axis="y", length=0)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
ax.spines["bottom"].set_color("#CCCCCC")
ax.spines["left"].set_color("#CCCCCC")

# --- Pie chart ---
n_confirmed = len(confirmed)
n_rejected = len(rejected) + len(tentative)
n_total = n_confirmed + n_rejected

pie_x = 0.5
pie_y = 0.5
pie_size = 0.25
pie_aspect = pie_size * (12 / fig_height)

ax_pie = fig.add_axes([pie_x, pie_y, pie_size, pie_aspect])

confirmed_pct = n_confirmed / n_total * 100
rejected_pct = n_rejected / n_total * 100

wedges, _ = ax_pie.pie(
    [n_confirmed, n_rejected],
    colors=[PC["color_confirmed"], PC["color_rejected"]],
    startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2.5),
)

ax_pie.text(-0.30, 0.30, f"{n_confirmed}\n({confirmed_pct:.1f}%)",
            ha="center", va="center", fontsize=12, fontweight="bold",
            color="white", zorder=5)
ax_pie.text(0.20, -0.20, f"{n_rejected}\n({rejected_pct:.1f}%)",
            ha="center", va="center", fontsize=12, fontweight="bold",
            color="white", zorder=5)

ax_pie.text(-0.75, 0.90, "Confirmed", fontsize=12, fontweight="bold",
            color=PC["color_text"], ha="center")
ax_pie.text(1.05, -0.75, "Rejected", fontsize=12, fontweight="bold",
            color=PC["color_text"], ha="center")

bbox_props = dict(boxstyle="round,pad=0.4", facecolor=PC["color_pie_label_bg"],
                  edgecolor="none", alpha=0.9)
ax_pie.text(0.0, -1.4, f"Total Features: {n_total}",
            ha="center", va="center", fontsize=13, fontweight="bold",
            color="#2B6D8E", bbox=bbox_props)

# Save results
plt.savefig("boruta_feature_selection_plot.tiff", dpi=PC["dpi"],
            bbox_inches="tight", facecolor="white",
            pil_kwargs={"compression": "tiff_lzw"})
plt.savefig("boruta_feature_selection_plot.png", dpi=PC["dpi"],
            bbox_inches="tight", facecolor="white")
plt.close()

print(f"  Figures saved:")
print(f"    boruta_feature_selection_plot.tiff (300 DPI)")
print(f"    boruta_feature_selection_plot.png")

# ============================================================
# 8b. Plot feature importance box plot (R-Boruta vertical style)
# ============================================================
# Equivalent to R's plot(boruta_result) output:
#   - Vertical boxes, feature names on X axis rotated 90°
#   - Importance on Y axis
#   - Green = Confirmed, Red = Rejected, Yellow = Tentative, Blue = Shadow
#   - Blue dashed line = Shadow Max median (decision reference line)
# ============================================================
print(f"\n{'='*70}")
print("STEP 5b: Plotting feature importance box plot (R-Boruta vertical style)")
print("=" * 70)

# --- Colours ---
COLOR_CONFIRMED_BOX = "#5BB37A"
COLOR_REJECTED_BOX  = "#E8637A"
COLOR_TENTATIVE_BOX = "#F5C842"
COLOR_SHADOW_BOX    = "#5B9BD5"

# --- Sort by meanImp descending ---
sort_idx = np.argsort(importance_mean)[::-1]
sorted_names   = [feature_names[i] for i in sort_idx]
sorted_support = boruta_selector.support_[sort_idx]
sorted_weak    = boruta_selector.support_weak_[sort_idx]
sorted_history = importance_history[:, sort_idx]

# --- Group indices ---
confirmed_indices = [i for i in range(len(sorted_names)) if sorted_support[i]]
tentative_indices = [i for i in range(len(sorted_names)) if sorted_weak[i]]
rejected_indices  = [i for i in range(len(sorted_names))
                     if not sorted_support[i] and not sorted_weak[i]]

# --- Limit number of displayed features ---
MAX_DISPLAY = 140
n_rejected_show = max(0, MAX_DISPLAY - len(confirmed_indices) - len(tentative_indices))
display_indices = confirmed_indices + tentative_indices + rejected_indices[:n_rejected_show]
display_indices.sort()

# --- Assemble plot data: shadow first, then features sorted high to low ---
box_data = [shadow_max_history]
box_labels = ["Shadow\nMax"]
box_face_colors = [COLOR_SHADOW_BOX]

for idx in display_indices:
    box_data.append(sorted_history[:, idx])
    box_labels.append(sorted_names[idx])
    if sorted_support[idx]:
        box_face_colors.append(COLOR_CONFIRMED_BOX)
    elif sorted_weak[idx]:
        box_face_colors.append(COLOR_TENTATIVE_BOX)
    else:
        box_face_colors.append(COLOR_REJECTED_BOX)

n_boxes = len(box_data)
x_positions = np.arange(n_boxes)

# --- Plot ---
fig_w_box = max(16, n_boxes * 0.42)
fig_box, ax_box = plt.subplots(figsize=(fig_w_box, 8), facecolor="white")
ax_box.set_facecolor("white")

bp = ax_box.boxplot(
    box_data,
    positions=x_positions,
    vert=True,                 # vertical boxes
    widths=0.55,
    patch_artist=True,
    showfliers=True,
    flierprops=dict(marker="o", markersize=2.5, alpha=0.3,
                    markerfacecolor="#999999", markeredgecolor="none"),
    medianprops=dict(color="#333333", linewidth=1.5),
    whiskerprops=dict(color="#666666", linewidth=0.8),
    capprops=dict(color="#666666", linewidth=0.8),
    boxprops=dict(linewidth=0.8),
)

# Box colours
for patch, color in zip(bp["boxes"], box_face_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
    patch.set_edgecolor("#555555")

# Shadow reference line: median of shadow max values → horizontal dashed line
shadow_median = np.median(shadow_max_history)
ax_box.axhline(y=shadow_median, color=COLOR_SHADOW_BOX, linewidth=1.2,
               linestyle="--", alpha=0.7, zorder=1)

# Vertical separator between shadow and feature boxes
ax_box.axvline(x=0.5, color="#CCCCCC", linewidth=0.8, linestyle="-", alpha=0.5)

# X axis labels (rotated 90°)
ax_box.set_xticks(x_positions)
ax_box.set_xticklabels(box_labels, fontsize=8.5, color=PC["color_text"],
                       rotation=90, ha="center", va="top")

# Colour X axis tick labels by Boruta decision
xtick_labels_box = ax_box.get_xticklabels()
for i, label in enumerate(xtick_labels_box):
    label.set_color(box_face_colors[i])
    if i == 0:
        label.set_fontweight("bold")

# Y axis
ax_box.set_ylabel("Importance", fontsize=13, fontweight="bold",
                  color=PC["color_text"], labelpad=10)
ax_box.set_xlabel("Features", fontsize=13, fontweight="bold",
                  color=PC["color_text"], labelpad=15)

# Grid
ax_box.yaxis.grid(True, linestyle="-", alpha=0.12, zorder=0)
ax_box.set_axisbelow(True)
ax_box.tick_params(axis="y", labelsize=10, colors=PC["color_text"])
ax_box.tick_params(axis="x", length=0)

for spine in ["top", "right"]:
    ax_box.spines[spine].set_visible(False)
ax_box.spines["bottom"].set_color("#CCCCCC")
ax_box.spines["left"].set_color("#CCCCCC")

# Legend
legend_patches = [
    mpatches.Patch(facecolor=COLOR_CONFIRMED_BOX, edgecolor="#555555",
                   alpha=0.75, label=f"Confirmed ({len(confirmed_indices)})"),
    mpatches.Patch(facecolor=COLOR_TENTATIVE_BOX, edgecolor="#555555",
                   alpha=0.75, label=f"Tentative ({len(tentative_indices)})"),
    mpatches.Patch(facecolor=COLOR_REJECTED_BOX, edgecolor="#555555",
                   alpha=0.75, label=f"Rejected ({len(rejected_indices)})"),
    mpatches.Patch(facecolor=COLOR_SHADOW_BOX, edgecolor="#555555",
                   alpha=0.75, label="Shadow Max (MZSA)"),
]
ax_box.legend(handles=legend_patches, loc="upper right", fontsize=10,
              frameon=True, framealpha=0.9, edgecolor="#CCCCCC")

plt.tight_layout()

# Save results
plt.savefig("boruta_importance_boxplot.tiff", dpi=PC["dpi"],
            bbox_inches="tight", facecolor="white",
            pil_kwargs={"compression": "tiff_lzw"})
plt.savefig("boruta_importance_boxplot.png", dpi=PC["dpi"],
            bbox_inches="tight", facecolor="white")
plt.close()

print(f"  Box plot saved:")
print(f"    boruta_importance_boxplot.tiff (300 DPI)")
print(f"    boruta_importance_boxplot.png")


  图片已保存:
    boruta_feature_selection_plot.tiff (300 DPI)
    boruta_feature_selection_plot.png

STEP 5b: 绘制特征重要性箱线图（R-Boruta 垂直样式）
  箱线图已保存:
    boruta_importance_boxplot.tiff (300 DPI)
    boruta_importance_boxplot.png


In [20]:
"""
Post-Feature-Selection Statistical Description
=================================================
Descriptive statistics for three datasets AFTER Boruta feature selection.

Expected Input Files (in ./Data/):
  boruta_results.csv                         Full Boruta results (meanImp, medianImp, normHits, Decision)
  selected_features.csv                      List of confirmed features
  data_training_selected.csv                 Training set (selected features only)
  data_internal_validation_selected.csv      Internal validation set (selected features only)
  data_temporal_validation_selected.csv      Temporal validation set (selected features only)

Excel Report Sheets (up to 12 sheets):
  1.  Feature Selection Summary  — Total vs selected, by type and group
  2.  Boruta Importance          — Full Boruta output: meanImp, medianImp, normHits, Decision
  3.  Dataset Overview           — Sample sizes, outcome, missingness (selected vars only)
  4.  Table 1 (Selected)         — Table 1 grouped by clinical domain, KW/Chi2 across 3 datasets
  5.  Continuous Variables       — Mean+/-SD, Median(IQR), Min-Max, KW test, SMD
  6.  Categorical Variables      — n(%) per category, Chi-square test, SMD
  7.  SMD Summary                — Training vs Validation balance for selected features
  8.  Variable Dictionary        — Selected variables: label, type, group, value range
  9.  Outcome Training           — SA+ vs SA- in Training set (MWU / Chi2 / Fisher)
  10. Outcome Internal Val       — SA+ vs SA- in Internal Validation set
  11. Outcome Temporal Val       — SA+ vs SA- in Temporal Validation set
  12. Outcome Summary            — All 3 datasets SA+ vs SA- side by side in one table

Dependencies:
  pip install pandas numpy scipy openpyxl
"""

import pandas as pd
import numpy as np
from scipy import stats
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings
import sys
import os

warnings.filterwarnings("ignore")

# ============================================================
# Global Configuration
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VAR = "BaseInfoDate"

# ---- File Paths (modify as needed) ----
DATA_DIR = "./Data"

BORUTA_RESULTS_PATH = os.path.join(DATA_DIR, "boruta_results.csv")
SELECTED_FEATURES_PATH = os.path.join(DATA_DIR, "selected_features.csv")

TRAIN_PATH = os.path.join(DATA_DIR, "data_training_selected.csv")
INTVAL_PATH = os.path.join(DATA_DIR, "data_internal_validation_selected.csv")
TEMPVAL_PATH = os.path.join(DATA_DIR, "data_temporal_validation_selected.csv")

OUTPUT_PATH = os.path.join(DATA_DIR, "statistical_description_selected.xlsx")

# ============================================================
# Variable Definitions (full set — used for classification)
# ============================================================
CONTINUOUS_VARS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
    "Hemoglobin", "RBC", "Platelet", "WBC",
    "NeutrophilPct", "LymphocytePct",
    "Glucose", "WifeALT", "WifeCreat", "TSH",
    "HusbALT", "HusbCreat",
    "LeftTestVol", "RightTestVol",
]
ORDINAL_VARS = ["WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc"]
NOMINAL_VARS = [
    "WifeEthnic", "HusbandEthnic", "WifeResident", "HusbandResident",
    "WifeABO", "HusbABO", "CycleRegular", "MenstrualVol", "Dysmenorrhea",
    "WifeMental", "WifeIntel", "WifeFace", "WifePosture", "WifeFaceSpec", "WifeSkinHair",
    "HusbMental", "HusbIntel", "HusbFace", "HusbPosture", "HusbFaceSpec", "HusbSkinHair",
    "WifePubHair", "Breast", "Vulva",
    "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
    "Testis", "Epididymis", "VasDeferens", "Varicocele",
    "WifeUrine", "HusbUrine", "WifeHepB", "HusbHepB",
]
BINARY_VARS = [
    "WifeDiseaseHx", "WifeBirthDefect", "GynDisease",
    "WifeMedication", "WifeVaccine", "Contraception",
    "PrevPregnancy", "Consanguinity", "WifeFamConsang", "WifeFamDisease",
    "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
    "WifeSmoke", "WifePassSmoke", "WifeAlcohol",
    "WifeDrugUse", "WifeHalitosis", "WifeGumBleed",
    "WifeEnvExpose", "WifeStress", "WifeRelatStress", "WifeFinance", "WifeReady",
    "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
    "WifeLiverSpleen", "WifeExtremity", "WifeGynExam",
    "WifeRh", "HusbRh",
    "RubellaIgG", "CMVIgG", "CMVIgM", "ToxoIgG", "ToxoIgM",
    "SyphilisScr", "HusbSyphilis", "GynUltrasound",
    "HusbDiseaseHis", "HusbBirthDefect", "HusbAndrologyDis",
    "HusbMedication", "HusbVaccinated",
    "HusbFamConsang", "HusbFamDisease",
    "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
    "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse",
    "HusbEnvExpose", "HusbStress", "HusbRelatStress", "HusbFinance", "HusbReady",
    "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
    "HusbLiverSpleen", "HusbExtremity", "HusbAndroExam",
]
ALL_VARS = CONTINUOUS_VARS + ORDINAL_VARS + NOMINAL_VARS + BINARY_VARS

VAR_LABELS = {
    "SpontAbortion": "Spontaneous Abortion",
    "WifeAge": "Wife Age (years)", "HusbandAge": "Husband Age (years)",
    "MenarcheAge": "Menarche Age (years)",
    "WifeHeight": "Wife Height (cm)", "WifeBMI": "Wife BMI (kg/m2)",
    "WifeHR": "Wife Heart Rate (bpm)", "WifeSBP": "Wife SBP (mmHg)", "WifeDBP": "Wife DBP (mmHg)",
    "HusbHeight": "Husband Height (cm)", "HusbBMI": "Husband BMI (kg/m2)",
    "HusbHR": "Husband Heart Rate (bpm)", "HusbSBP": "Husband SBP (mmHg)", "HusbDBP": "Husband DBP (mmHg)",
    "Hemoglobin": "Hemoglobin (g/L)", "RBC": "RBC (x10^12/L)",
    "Platelet": "Platelet (x10^9/L)", "WBC": "WBC (x10^9/L)",
    "NeutrophilPct": "Neutrophil %", "LymphocytePct": "Lymphocyte %",
    "Glucose": "Fasting Glucose (mmol/L)", "WifeALT": "Wife ALT (U/L)", "WifeCreat": "Wife Creatinine (umol/L)",
    "TSH": "TSH (mIU/L)", "HusbALT": "Husband ALT (U/L)", "HusbCreat": "Husband Creatinine (umol/L)",
    "LeftTestVol": "Left Testicular Vol (ml)", "RightTestVol": "Right Testicular Vol (ml)",
    "WifeEdu": "Wife Education", "HusbandEdu": "Husband Education",
    "WifeOcc": "Wife Occupation", "HusbandOcc": "Husband Occupation",
    "WifeEthnic": "Wife Ethnicity", "HusbandEthnic": "Husband Ethnicity",
    "WifeResident": "Wife Residency", "HusbandResident": "Husband Residency",
    "WifeABO": "Wife ABO Blood Type", "HusbABO": "Husband ABO Blood Type",
    "CycleRegular": "Menstrual Regularity", "MenstrualVol": "Menstrual Volume", "Dysmenorrhea": "Dysmenorrhea",
    "WifeMental": "Wife Mental Status", "WifeIntel": "Wife Intelligence",
    "WifeFace": "Wife Facial Appearance", "WifePosture": "Wife Posture",
    "WifeFaceSpec": "Wife Facial Features", "WifeSkinHair": "Wife Skin & Hair",
    "HusbMental": "Husband Mental Status", "HusbIntel": "Husband Intelligence",
    "HusbFace": "Husband Facial Appearance", "HusbPosture": "Husband Posture",
    "HusbFaceSpec": "Husband Facial Features", "HusbSkinHair": "Husband Skin & Hair",
    "WifePubHair": "Female Pubic Hair", "Breast": "Breast Development", "Vulva": "Vulva",
    "HusbPubHair": "Male Pubic Hair", "AdamsApple": "Adam's Apple", "Penis": "Penis",
    "Foreskin": "Foreskin", "Testis": "Testis", "Epididymis": "Epididymis",
    "VasDeferens": "Vas Deferens", "Varicocele": "Varicocele",
    "WifeUrine": "Wife Urinalysis", "HusbUrine": "Husband Urinalysis",
    "WifeHepB": "Wife Hepatitis B", "HusbHepB": "Husband Hepatitis B",
    "WifeDiseaseHx": "Wife Disease History", "WifeBirthDefect": "Wife Birth Defect Hx",
    "GynDisease": "Gynecological Disease", "WifeMedication": "Wife Medication Hx",
    "WifeVaccine": "Wife Vaccination", "Contraception": "Contraception Hx",
    "PrevPregnancy": "Previous Pregnancy", "Consanguinity": "Consanguineous Marriage",
    "WifeFamConsang": "Wife Family Consanguinity", "WifeFamDisease": "Wife Family Disease Hx",
    "WifeMeatEgg": "Wife Meat/Egg Intake", "WifeVegAverse": "Wife Vegetable Aversion",
    "WifeRawMeat": "Wife Raw Meat", "WifeSmoke": "Wife Smoking",
    "WifePassSmoke": "Wife Passive Smoking", "WifeAlcohol": "Wife Alcohol",
    "WifeDrugUse": "Wife Drug Use", "WifeHalitosis": "Wife Halitosis",
    "WifeGumBleed": "Wife Gum Bleeding", "WifeEnvExpose": "Wife Env Exposure",
    "WifeStress": "Wife Work Stress", "WifeRelatStress": "Wife Relationship Stress",
    "WifeFinance": "Wife Financial Stress", "WifeReady": "Wife Reproductive Readiness",
    "WifeThyroid": "Wife Thyroid Exam", "WifeLung": "Wife Lung Exam",
    "WifeRhythm": "Wife Cardiac Rhythm", "WifeMurmur": "Wife Heart Murmur",
    "WifeLiverSpleen": "Wife Liver/Spleen", "WifeExtremity": "Wife Extremity Exam",
    "WifeGynExam": "Gynecological Exam", "WifeRh": "Wife Rh Type", "HusbRh": "Husband Rh Type",
    "RubellaIgG": "Rubella IgG", "CMVIgG": "CMV IgG", "CMVIgM": "CMV IgM",
    "ToxoIgG": "Toxoplasma IgG", "ToxoIgM": "Toxoplasma IgM",
    "SyphilisScr": "Wife Syphilis Screen", "HusbSyphilis": "Husband Syphilis Screen",
    "GynUltrasound": "Gyn Ultrasound",
    "HusbDiseaseHis": "Husband Disease Hx", "HusbBirthDefect": "Husband Birth Defect Hx",
    "HusbAndrologyDis": "Andrological Disease", "HusbMedication": "Husband Medication Hx",
    "HusbVaccinated": "Husband Vaccination",
    "HusbFamConsang": "Husband Family Consanguinity", "HusbFamDisease": "Husband Family Disease Hx",
    "HusbMeatEgg": "Husband Meat/Egg Intake", "HusbVegAverse": "Husband Vegetable Aversion",
    "HusbRawMeat": "Husband Raw Meat", "HusbSmoke": "Husband Smoking",
    "HusbPassSmoke": "Husband Passive Smoking", "HusbAlcohol": "Husband Alcohol",
    "HusbDrugUse": "Husband Drug Use", "HusbEnvExpose": "Husband Env Exposure",
    "HusbStress": "Husband Work Stress", "HusbRelatStress": "Husband Relationship Stress",
    "HusbFinance": "Husband Financial Stress", "HusbReady": "Husband Reproductive Readiness",
    "HusbThyroid": "Husband Thyroid Exam", "HusbLung": "Husband Lung Exam",
    "HusbRhythm": "Husband Cardiac Rhythm", "HusbMurmur": "Husband Heart Murmur",
    "HusbLiverSpleen": "Husband Liver/Spleen", "HusbExtremity": "Husband Extremity Exam",
    "HusbAndroExam": "Andrological Exam",
}

VAR_GROUPS = {
    "Demographics": [
        "WifeAge", "HusbandAge", "WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc",
        "WifeEthnic", "HusbandEthnic", "WifeResident", "HusbandResident"],
    "Wife Physical Exam": [
        "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
        "WifeMental", "WifeIntel", "WifeFace", "WifePosture", "WifeFaceSpec", "WifeSkinHair",
        "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur", "WifeLiverSpleen", "WifeExtremity"],
    "Husband Physical Exam": [
        "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
        "HusbMental", "HusbIntel", "HusbFace", "HusbPosture", "HusbFaceSpec", "HusbSkinHair",
        "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur", "HusbLiverSpleen", "HusbExtremity"],
    "Menstruation & Reproduction": [
        "MenarcheAge", "CycleRegular", "MenstrualVol", "Dysmenorrhea",
        "PrevPregnancy", "Contraception", "WifeGynExam", "GynUltrasound",
        "WifePubHair", "Breast", "Vulva", "GynDisease"],
    "Andrological Exam": [
        "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
        "Testis", "Epididymis", "VasDeferens", "Varicocele",
        "LeftTestVol", "RightTestVol", "HusbAndroExam", "HusbAndrologyDis"],
    "Hematology": ["Hemoglobin", "RBC", "Platelet", "WBC", "NeutrophilPct", "LymphocytePct"],
    "Biochemistry & Hormones": ["Glucose", "WifeALT", "WifeCreat", "HusbALT", "HusbCreat", "TSH"],
    "Blood Type": ["WifeABO", "HusbABO", "WifeRh", "HusbRh"],
    "Infection Screening": [
        "RubellaIgG", "CMVIgG", "CMVIgM", "ToxoIgG", "ToxoIgM",
        "SyphilisScr", "HusbSyphilis", "WifeHepB", "HusbHepB", "WifeUrine", "HusbUrine"],
    "Medical & Family History": [
        "WifeDiseaseHx", "WifeBirthDefect", "WifeMedication", "WifeVaccine",
        "WifeFamConsang", "WifeFamDisease", "Consanguinity",
        "HusbDiseaseHis", "HusbBirthDefect", "HusbMedication", "HusbVaccinated",
        "HusbFamConsang", "HusbFamDisease"],
    "Lifestyle (Wife)": [
        "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
        "WifeSmoke", "WifePassSmoke", "WifeAlcohol", "WifeDrugUse", "WifeHalitosis", "WifeGumBleed"],
    "Lifestyle (Husband)": [
        "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
        "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse"],
    "Psychosocial Factors": [
        "WifeEnvExpose", "WifeStress", "WifeRelatStress", "WifeFinance", "WifeReady",
        "HusbEnvExpose", "HusbStress", "HusbRelatStress", "HusbFinance", "HusbReady"],
}

# ---- Category Value Labels (coded value -> display label) ----
# Education: shared by WifeEdu and HusbandEdu
_EDU_LABELS = {
    "A": "Illiterate", "B": "Elementary School", "C": "Junior High School",
    "D": "High School / Vocational", "E": "Bachelor's Degree", "F": "Graduate Degree",
}
# Occupation: shared by WifeOcc and HusbandOcc
_OCC_LABELS = {
    "A": "Farmer", "B": "Worker", "C": "Teacher / Civil Servant / Office",
    "D": "Service Industry", "E": "Business", "F": "Housework",
}
# ABO blood type: WifeABO uses raw category names, HusbABO uses A/B/C/D codes
_WIFE_ABO_LABELS = {
    "Type O": "Type O", "Type A": "Type A", "Type B": "Type B", "Type AB": "Type AB",
}
_HUSB_ABO_LABELS = {
    "A": "Type O", "B": "Type A", "C": "Type B", "D": "Type AB",
}
# Hepatitis B: shared by WifeHepB and HusbHepB
_HEPB_LABELS = {
    "A": "HBeAg Positive",
    "B": "Minor Hepatitis B",
    "C": "Past Infection",
    "D": "Recovery / Vaccination",
    "E": "Susceptible Population",
}

VALUE_LABELS = {
    "WifeEdu": _EDU_LABELS,
    "HusbandEdu": _EDU_LABELS,
    "WifeOcc": _OCC_LABELS,
    "HusbandOcc": _OCC_LABELS,
    "WifeABO": _WIFE_ABO_LABELS,
    "HusbABO": _HUSB_ABO_LABELS,
    "WifeHepB": _HEPB_LABELS,
    "HusbHepB": _HEPB_LABELS,
    # Add more variable value-label mappings here as needed
}

def cat_label(var, raw_value):
    """Translate a coded category value to its display label. Returns original if no mapping."""
    raw_str = str(raw_value).strip()
    if var in VALUE_LABELS and raw_str in VALUE_LABELS[var]:
        return VALUE_LABELS[var][raw_str]
    return raw_str

def cat_sort_key(var, cat_str):
    """Sort key for categories: use original coded order if value labels exist."""
    if var in VALUE_LABELS:
        order = list(VALUE_LABELS[var].keys())
        raw = str(cat_str).strip()
        # Check if cat_str is a label (already translated) — reverse lookup
        for code, label in VALUE_LABELS[var].items():
            if label == raw:
                try: return order.index(code)
                except ValueError: pass
        # Check if cat_str is a raw code
        try: return order.index(raw)
        except ValueError: pass
    return str(cat_str)

DS_TRAIN = "Training"
DS_INTVAL = "Internal Validation"
DS_TEMPVAL = "Temporal Validation"


# ============================================================
# Helpers
# ============================================================
def safe_num(s): return pd.to_numeric(s, errors="coerce")
def smd_cont(x1, x2):
    p = np.sqrt((np.nanstd(x1,ddof=1)**2 + np.nanstd(x2,ddof=1)**2)/2)
    return abs(np.nanmean(x1)-np.nanmean(x2))/p if p>0 else 0.0
def smd_bin(p1, p2):
    p = np.sqrt((p1*(1-p1)+p2*(1-p2))/2)
    return abs(p1-p2)/p if p>0 else 0.0
def fmt_p(p):
    if pd.isna(p): return ""
    return "<0.001" if p<0.001 else f"{p:.3f}"
def outcome_01(df):
    if OUTCOME_VAR not in df.columns: return pd.Series(dtype=float)
    return df[OUTCOME_VAR].map({True:1,False:0,"TRUE":1,"FALSE":0,"True":1,"False":0,1:1,0:0}).astype(float)
def var_type(v):
    if v in CONTINUOUS_VARS: return "Continuous"
    if v in ORDINAL_VARS: return "Ordinal"
    if v in NOMINAL_VARS: return "Nominal"
    if v in BINARY_VARS: return "Binary"
    return "Unknown"
def var_group(v):
    for g, vs in VAR_GROUPS.items():
        if v in vs: return g
    return "Other"

def classify(selected):
    s = set(selected)
    return ([v for v in CONTINUOUS_VARS if v in s],
            [v for v in ORDINAL_VARS if v in s],
            [v for v in NOMINAL_VARS if v in s],
            [v for v in BINARY_VARS if v in s])

def sel_groups(selected):
    s = set(selected)
    return {g: [v for v in vs if v in s] for g, vs in VAR_GROUPS.items() if any(v in s for v in vs)}


# ============================================================
# Core Statistics
# ============================================================
def desc_cont(datasets, var):
    res = {}; vals_list = []
    for nm, df in datasets.items():
        if var not in df.columns: continue
        v = safe_num(df[var]).dropna(); vals_list.append(v)
        n = len(v); m = len(df)-n
        if n>0:
            res[nm] = dict(N=n, Miss=m, MissP=m/len(df)*100,
                Mean=np.mean(v), SD=np.std(v,ddof=1), Med=np.median(v),
                Q1=np.percentile(v,25), Q3=np.percentile(v,75),
                Min=np.min(v), Max=np.max(v),
                MeanSD=f"{np.mean(v):.2f} \u00b1 {np.std(v,ddof=1):.2f}",
                MedIQR=f"{np.median(v):.2f} ({np.percentile(v,25):.2f}-{np.percentile(v,75):.2f})")
        else:
            res[nm] = dict(N=0,Miss=m,MissP=m/len(df)*100,Mean=np.nan,SD=np.nan,
                Med=np.nan,Q1=np.nan,Q3=np.nan,Min=np.nan,Max=np.nan,MeanSD="—",MedIQR="—")
    pk = np.nan
    vv = [x for x in vals_list if len(x)>0]
    if len(vv)>=2:
        try: _, pk = stats.kruskal(*vv)
        except: pass
    nms = list(datasets.keys()); si=st=np.nan
    if len(nms)>=2:
        a = safe_num(datasets[nms[0]][var]).dropna() if var in datasets[nms[0]].columns else pd.Series()
        b = safe_num(datasets[nms[1]][var]).dropna() if var in datasets[nms[1]].columns else pd.Series()
        if len(a)>0 and len(b)>0: si = smd_cont(a, b)
    if len(nms)>=3:
        c = safe_num(datasets[nms[2]][var]).dropna() if var in datasets[nms[2]].columns else pd.Series()
        if len(a)>0 and len(c)>0: st = smd_cont(a, c)
    return res, pk, si, st

def desc_cat(datasets, var):
    res = {}; cats = set()
    for nm, df in datasets.items():
        if var not in df.columns: continue
        s = df[var].map({True:"True",False:"False",1:"True",0:"False"}).fillna(df[var])
        vc = s.value_counts(dropna=True); n = vc.sum(); m = df[var].isna().sum()
        cats.update(vc.index)
        res[nm] = dict(N=n, Miss=m, MissP=m/len(df)*100 if len(df)>0 else 0, cts=vc, pcts=vc/n*100 if n>0 else vc*0)
    cats = sorted(cats, key=lambda x: cat_sort_key(var, x))
    pc = np.nan; nms = list(datasets.keys())
    if len(nms)>=2 and len(cats)>=2:
        try:
            ct = pd.DataFrame(index=cats)
            for nm in nms:
                if nm in res: ct[nm] = res[nm]["cts"].reindex(cats, fill_value=0)
            _, pc, _, _ = stats.chi2_contingency(ct.values)
        except: pass
    si=st=np.nan
    if len(cats)==2 and len(nms)>=2:
        tc = "True" if "True" in cats else cats[-1]
        pp = [res[nm]["cts"].get(tc,0)/res[nm]["N"] if nm in res and res[nm]["N"]>0 else np.nan for nm in nms]
        if len(pp)>=2 and not np.isnan(pp[0]) and not np.isnan(pp[1]): si = smd_bin(pp[0], pp[1])
        if len(pp)>=3 and not np.isnan(pp[0]) and not np.isnan(pp[2]): st = smd_bin(pp[0], pp[2])
    return res, cats, pc, si, st


# ============================================================
# Excel Styles
# ============================================================
class S:
    hf = Font(bold=True, size=11, color="FFFFFF", name="Arial")
    hfl = PatternFill("solid", fgColor="4472C4")
    sf = PatternFill("solid", fgColor="E2EFDA")
    sft = Font(bold=True, size=11, name="Arial", color="375623")
    # Decision color fills
    conf_fill = PatternFill("solid", fgColor="C6EFCE")   # green for Confirmed
    tent_fill = PatternFill("solid", fgColor="FFEB9C")   # yellow for Tentative
    rej_fill  = PatternFill("solid", fgColor="FFC7CE")   # red for Rejected
    nf = Font(size=10, name="Arial")
    bf = Font(size=10, name="Arial", bold=True)
    rf = Font(size=10, name="Arial", bold=True, color="C00000")
    gf = Font(size=10, name="Arial", bold=True, color="006100")
    it = Font(size=10, name="Arial", italic=True, color="808080")
    nt = Font(size=9, name="Arial", italic=True, color="666666")
    bd = Border(left=Side(style="thin",color="BFBFBF"),right=Side(style="thin",color="BFBFBF"),
                top=Side(style="thin",color="BFBFBF"),bottom=Side(style="thin",color="BFBFBF"))
    ct = Alignment(horizontal="center")
    wc = Alignment(horizontal="center", wrap_text=True)
    lt = Alignment(horizontal="left")

def wh(ws, r, hds):
    for c, h in enumerate(hds, 1):
        cl = ws.cell(row=r, column=c, value=h)
        cl.font=S.hf; cl.fill=S.hfl; cl.alignment=S.wc; cl.border=S.bd

def wc(ws, r, c, v, font=None, fill=None, align=None):
    cl = ws.cell(row=r, column=c, value=v)
    cl.font=font or S.nf; cl.border=S.bd; cl.alignment=align or S.ct
    if fill: cl.fill=fill
    return cl


# ============================================================
# Sheet 1: Feature Selection Summary
# ============================================================
def write_selection_summary(wb, selected, sc, so, sn, sb, sg):
    ws = wb.active; ws.title = "Feature Selection Summary"
    wh(ws, 1, ["Category", "Total Available", "Selected", "Selection Rate"])
    data = [("All Variables", len(ALL_VARS), len(selected)),
            ("  Continuous", len(CONTINUOUS_VARS), len(sc)),
            ("  Ordinal", len(ORDINAL_VARS), len(so)),
            ("  Nominal", len(NOMINAL_VARS), len(sn)),
            ("  Binary", len(BINARY_VARS), len(sb))]
    r = 2
    for lb, tot, sel in data:
        wc(ws,r,1,lb,font=S.bf,align=S.lt); wc(ws,r,2,tot); wc(ws,r,3,sel)
        wc(ws,r,4,f"{sel/tot*100:.1f}%" if tot>0 else "—"); r+=1

    r+=1; wc(ws,r,1,"By Clinical Group",font=S.bf,fill=S.sf,align=S.lt)
    for c in range(2,5): wc(ws,r,c,"",fill=S.sf)
    r+=1; wh(ws,r,["Group","Total Available","Selected","Selection Rate"]); r+=1
    for gn, gv in VAR_GROUPS.items():
        tg=len(gv); sg_=len([v for v in gv if v in set(selected)])
        wc(ws,r,1,gn,align=S.lt); wc(ws,r,2,tg); wc(ws,r,3,sg_)
        wc(ws,r,4,f"{sg_/tg*100:.1f}%" if tg>0 else "—"); r+=1

    r+=1; wc(ws,r,1,"Selected Features List",font=S.bf,fill=S.sf,align=S.lt)
    for c in range(2,5): wc(ws,r,c,"",fill=S.sf)
    r+=1; wh(ws,r,["No.","Variable","Label","Type"]); r+=1
    for i, v in enumerate(selected, 1):
        wc(ws,r,1,i); wc(ws,r,2,v,font=S.bf,align=S.lt)
        wc(ws,r,3,VAR_LABELS.get(v,""),align=S.lt); wc(ws,r,4,var_type(v)); r+=1
    ws.column_dimensions["A"].width=30; ws.column_dimensions["B"].width=18
    ws.column_dimensions["C"].width=30; ws.column_dimensions["D"].width=14
    ws.freeze_panes="A2"


# ============================================================
# Sheet 2: Boruta Importance (from boruta_results.csv)
# ============================================================
def write_boruta_importance(wb, boruta_df, selected_set):
    ws = wb.create_sheet("Boruta Importance")

    # Identify columns
    col_map = {c.lower().strip(): c for c in boruta_df.columns}
    var_col = None
    for k in ["variable", "feature", "var", "name"]:
        if k in col_map: var_col = col_map[k]; break
    if var_col is None: var_col = boruta_df.columns[0]

    dec_col = col_map.get("decision", None)

    # Determine importance metric columns
    imp_cols = []
    for orig_col in boruta_df.columns:
        if orig_col in (var_col, dec_col): continue
        imp_cols.append(orig_col)

    # Build header
    headers = ["No.", "Variable", "Label", "Type", "Group"]
    headers += imp_cols
    if dec_col: headers.append("Decision")
    headers.append("Selected")
    wh(ws, 1, headers)

    # Sort by first importance column descending (usually meanImp)
    df_sorted = boruta_df.copy()
    if imp_cols:
        first_imp = imp_cols[0]
        df_sorted[first_imp] = pd.to_numeric(df_sorted[first_imp], errors="coerce")
        df_sorted = df_sorted.sort_values(first_imp, ascending=False, na_position="last")

    r = 2
    for idx, (_, row_data) in enumerate(df_sorted.iterrows(), 1):
        vname = str(row_data[var_col]).strip()
        wc(ws, r, 1, idx)
        wc(ws, r, 2, vname, font=S.bf, align=S.lt)
        wc(ws, r, 3, VAR_LABELS.get(vname, ""), align=S.lt)
        wc(ws, r, 4, var_type(vname))
        wc(ws, r, 5, var_group(vname), align=S.lt)

        c = 6
        for ic in imp_cols:
            val = row_data[ic]
            if pd.notna(val):
                try: wc(ws, r, c, round(float(val), 4))
                except: wc(ws, r, c, str(val))
            else:
                wc(ws, r, c, "")
            c += 1

        if dec_col:
            decision = str(row_data[dec_col]).strip()
            dec_fill = None
            if decision.lower() in ("confirmed", "accepted", "selected"): dec_fill = S.conf_fill
            elif decision.lower() == "tentative": dec_fill = S.tent_fill
            elif decision.lower() == "rejected": dec_fill = S.rej_fill
            wc(ws, r, c, decision, fill=dec_fill); c += 1

        is_sel = "Yes" if vname in selected_set else "No"
        wc(ws, r, c, is_sel, font=S.gf if is_sel=="Yes" else S.rf)
        r += 1

    ws.column_dimensions["A"].width=6; ws.column_dimensions["B"].width=22
    ws.column_dimensions["C"].width=28; ws.column_dimensions["D"].width=12
    ws.column_dimensions["E"].width=22
    for i, ic in enumerate(imp_cols):
        ws.column_dimensions[get_column_letter(6+i)].width=14
    if dec_col: ws.column_dimensions[get_column_letter(6+len(imp_cols))].width=14
    ws.column_dimensions[get_column_letter(6+len(imp_cols)+(1 if dec_col else 0))].width=10
    ws.freeze_panes="C2"
    ws.auto_filter.ref = f"A1:{get_column_letter(len(headers))}{r-1}"


# ============================================================
# Sheet 3: Dataset Overview
# ============================================================
def write_overview(wb, datasets, selected):
    ws = wb.create_sheet("Dataset Overview"); dn = list(datasets.keys())
    wh(ws, 1, ["Metric"] + dn); r = 2
    sc, so, sn, sb = classify(selected)

    rows = [("Sample Size (N)", [f"{len(datasets[n]):,}" for n in dn])]
    sa = []; neg = []
    for n in dn:
        y = outcome_01(datasets[n])
        sa.append(f"{int(y.sum()):,} ({y.mean()*100:.2f}%)")
        neg.append(f"{int(len(datasets[n])-y.sum()):,} ({(1-y.mean())*100:.2f}%)")
    rows += [("SA Positive n(%)", sa), ("SA Negative n(%)", neg)]
    rows.append(("Selected Features", [str(len([v for v in selected if v in datasets[n].columns])) for n in dn]))
    rows.append(("  Continuous", [str(len([v for v in sc if v in datasets[n].columns])) for n in dn]))
    rows.append(("  Categorical", [str(len([v for v in so+sn+sb if v in datasets[n].columns])) for n in dn]))
    mr = []
    for n in dn:
        ac=[c for c in selected if c in datasets[n].columns]; t=len(datasets[n])*len(ac) if ac else 1
        m=datasets[n][ac].isna().sum().sum(); mr.append(f"{int(m):,} ({m/t*100:.2f}%)")
    rows.append(("Missing Cells n(%)", mr))
    for lb, vs in rows:
        wc(ws,r,1,lb,font=S.bf,align=S.lt)
        for i, v in enumerate(vs): wc(ws,r,2+i,v)
        r+=1
    ws.column_dimensions["A"].width=25
    for i in range(len(dn)): ws.column_dimensions[get_column_letter(2+i)].width=26
    ws.freeze_panes="B2"


# ============================================================
# Sheet 4: Table 1
# ============================================================
def write_table1(wb, datasets, selected, sgrps):
    ws = wb.create_sheet("Table 1 (Selected)"); dn = list(datasets.keys())
    hds = ["Group","Variable","Label"]
    for n in dn: hds.append(f"{n} (N={len(datasets[n]):,})")
    hds += ["P-value","Test"]; wh(ws,1,hds)
    pc_=4+len(dn); tc_=pc_+1; r=2

    for gn, gvs in sgrps.items():
        wc(ws,r,1,gn,font=S.sft,fill=S.sf,align=S.lt)
        for c in range(2,tc_+1): wc(ws,r,c,"",fill=S.sf)
        r+=1
        for var in gvs:
            if not any(var in datasets[n].columns for n in dn): continue
            lb = VAR_LABELS.get(var,"")
            if var in CONTINUOUS_VARS:
                rs, pk, _, _ = desc_cont(datasets, var)
                wc(ws,r,1,""); wc(ws,r,2,var,font=S.bf,align=S.lt); wc(ws,r,3,lb,align=S.lt)
                col=4
                for n in dn: wc(ws,r,col,rs[n]["MedIQR"] if n in rs else "—"); col+=1
                p=wc(ws,r,pc_,fmt_p(pk))
                if not np.isnan(pk) and pk<0.05: p.font=S.rf
                wc(ws,r,tc_,"KW"); r+=1
            elif var in BINARY_VARS:
                rs, cats, pchi, _, _ = desc_cat(datasets, var)
                tcat = "True" if "True" in cats else (cats[-1] if cats else None)
                wc(ws,r,1,""); wc(ws,r,2,var,font=S.bf,align=S.lt); wc(ws,r,3,lb,align=S.lt)
                col=4
                for n in dn:
                    if n in rs and rs[n]["N"]>0 and tcat:
                        cn=rs[n]["cts"].get(tcat,0); pt=rs[n]["pcts"].get(tcat,0)
                        wc(ws,r,col,f"{int(cn):,} ({pt:.1f}%)")
                    else: wc(ws,r,col,"—")
                    col+=1
                p=wc(ws,r,pc_,fmt_p(pchi))
                if not np.isnan(pchi) and pchi<0.05: p.font=S.rf
                wc(ws,r,tc_,"Chi2"); r+=1
            else:
                rs, cats, pchi, _, _ = desc_cat(datasets, var)
                wc(ws,r,1,""); wc(ws,r,2,var,font=S.bf,align=S.lt); wc(ws,r,3,lb,align=S.lt)
                for c in range(4,4+len(dn)): wc(ws,r,c,"")
                p=wc(ws,r,pc_,fmt_p(pchi))
                if not np.isnan(pchi) and pchi<0.05: p.font=S.rf
                wc(ws,r,tc_,"Chi2"); r+=1
                for cat in cats:
                    wc(ws,r,1,""); wc(ws,r,2,""); wc(ws,r,3,f"  {cat_label(var, cat)}",align=S.lt)
                    col=4
                    for n in dn:
                        if n in rs and rs[n]["N"]>0:
                            cn=rs[n]["cts"].get(cat,0); pt=rs[n]["pcts"].get(cat,0)
                            wc(ws,r,col,f"{int(cn):,} ({pt:.1f}%)")
                        else: wc(ws,r,col,"—")
                        col+=1
                    wc(ws,r,pc_,""); wc(ws,r,tc_,""); r+=1

    r+=1
    note=("Note: Continuous vars as Median(IQR), P from Kruskal-Wallis; "
          "Binary vars as n(%) positive; Multi-category as n(%) per level, P from Chi-square. "
          "Only Boruta-confirmed features shown.")
    wc(ws,r,1,note,font=S.nt,align=S.lt)
    ws.merge_cells(start_row=r,start_column=1,end_row=r,end_column=tc_)
    ws.column_dimensions["A"].width=20; ws.column_dimensions["B"].width=22; ws.column_dimensions["C"].width=30
    for i in range(len(dn)): ws.column_dimensions[get_column_letter(4+i)].width=26
    ws.column_dimensions[get_column_letter(pc_)].width=12; ws.column_dimensions[get_column_letter(tc_)].width=8
    ws.freeze_panes="D2"


# ============================================================
# Sheet 5: Continuous Variables
# ============================================================
def write_continuous(wb, datasets, sc):
    ws = wb.create_sheet("Continuous Variables"); dn = list(datasets.keys())
    hds = ["Variable","Label"]
    for n in dn: hds += [f"{n} N(Miss%)",f"{n} Mean\u00b1SD",f"{n} Median(IQR)",f"{n} Min-Max"]
    hds += ["P (KW)","SMD(Train vs IntVal)","SMD(Train vs TempVal)"]
    wh(ws,1,hds); r=2
    for var in sc:
        if not any(var in datasets[n].columns for n in dn): continue
        rs, pk, si, st = desc_cont(datasets, var)
        wc(ws,r,1,var,font=S.bf,align=S.lt); wc(ws,r,2,VAR_LABELS.get(var,""),align=S.lt)
        col=3
        for n in dn:
            if n in rs:
                d=rs[n]; wc(ws,r,col,f"{d['N']:,} ({d['MissP']:.1f}%)")
                wc(ws,r,col+1,d["MeanSD"]); wc(ws,r,col+2,d["MedIQR"])
                wc(ws,r,col+3,f"{d['Min']:.2f}-{d['Max']:.2f}")
            else:
                for o in range(4): wc(ws,r,col+o,"—")
            col+=4
        p=wc(ws,r,col,fmt_p(pk))
        if not np.isnan(pk) and pk<0.05: p.font=S.rf
        col+=1
        for sv in [si,st]:
            c=wc(ws,r,col,f"{sv:.3f}" if not np.isnan(sv) else "")
            if not np.isnan(sv) and sv>0.1: c.font=S.rf
            col+=1
        r+=1
    ws.column_dimensions["A"].width=18; ws.column_dimensions["B"].width=28
    for i in range(2,len(hds)): ws.column_dimensions[get_column_letter(i+1)].width=24
    ws.freeze_panes="C2"


# ============================================================
# Sheet 6: Categorical Variables
# ============================================================
def write_categorical(wb, datasets, so, sn, sb):
    ws = wb.create_sheet("Categorical Variables"); dn = list(datasets.keys())
    hds = ["Variable","Category"]
    for n in dn: hds.append(f"{n} n(%)")
    hds += ["P (Chi2)","SMD(Train vs IntVal)","SMD(Train vs TempVal)"]
    wh(ws,1,hds); pc_=3+len(dn); r=2

    for var in sb+so+sn:
        if not any(var in datasets[n].columns for n in dn): continue
        rs, cats, pchi, si, st = desc_cat(datasets, var)
        wc(ws,r,1,var,font=S.sft,fill=S.sf,align=S.lt)
        wc(ws,r,2,VAR_LABELS.get(var,""),font=Font(size=10,name="Arial",italic=True),fill=S.sf)
        for cc in range(3,pc_): wc(ws,r,cc,"",fill=S.sf)
        p=wc(ws,r,pc_,fmt_p(pchi))
        if not np.isnan(pchi) and pchi<0.05: p.font=S.rf
        s1=wc(ws,r,pc_+1,f"{si:.3f}" if not np.isnan(si) else "")
        if not np.isnan(si) and si>0.1: s1.font=S.rf
        s2=wc(ws,r,pc_+2,f"{st:.3f}" if not np.isnan(st) else "")
        if not np.isnan(st) and st>0.1: s2.font=S.rf
        r+=1
        # Missing row
        wc(ws,r,1,""); wc(ws,r,2,"Missing n(%)",font=S.it)
        for di,n in enumerate(dn):
            if n in rs:
                mn=rs[n]["Miss"]; mp=rs[n]["MissP"]
                wc(ws,r,3+di,f"{mn:,} ({mp:.1f}%)" if mn>0 else "0 (0.0%)",font=S.rf if mn>0 else S.nf)
            else: wc(ws,r,3+di,"—")
        for cc in range(pc_,pc_+3): wc(ws,r,cc,""); r+=1
        for cat in cats:
            wc(ws,r,1,""); wc(ws,r,2,cat_label(var, cat),align=Alignment(horizontal="left",indent=2))
            for di,n in enumerate(dn):
                if n in rs and rs[n]["N"]>0:
                    cn=rs[n]["cts"].get(cat,0); pt=rs[n]["pcts"].get(cat,0)
                    wc(ws,r,3+di,f"{int(cn):,} ({pt:.1f}%)")
                else: wc(ws,r,3+di,"—")
            for cc in range(pc_,pc_+3): wc(ws,r,cc,""); r+=1

    ws.column_dimensions["A"].width=22; ws.column_dimensions["B"].width=28
    for i in range(len(dn)): ws.column_dimensions[get_column_letter(3+i)].width=22
    for i in range(3): ws.column_dimensions[get_column_letter(pc_+i)].width=22
    ws.freeze_panes="C2"


# ============================================================
# Sheet 7: SMD Summary
# ============================================================
def write_smd(wb, datasets, sc, sb):
    ws = wb.create_sheet("SMD Summary"); dn = list(datasets.keys())
    wh(ws,1,["Variable","Type","SMD (Train vs IntVal)","SMD (Train vs TempVal)","Balanced (<0.1)?"])
    r=2; recs=[]
    for v in sc:
        if not any(v in datasets[n].columns for n in dn): continue
        _,_,si,st = desc_cont(datasets,v); recs.append((v,"Continuous",si,st))
    for v in sb:
        if not any(v in datasets[n].columns for n in dn): continue
        _,_,_,si,st = desc_cat(datasets,v); recs.append((v,"Binary",si,st))
    for v,vt,s1,s2 in recs:
        wc(ws,r,1,v,align=S.lt); wc(ws,r,2,vt)
        for ci,sv in [(3,s1),(4,s2)]:
            c=wc(ws,r,ci,f"{sv:.4f}" if not np.isnan(sv) else "")
            if not np.isnan(sv) and sv>0.1: c.font=S.rf
        bal="Yes" if (not np.isnan(s1) and s1<=0.1 and not np.isnan(s2) and s2<=0.1) else "No"
        if np.isnan(s1) or np.isnan(s2): bal="N/A"
        wc(ws,r,5,bal,font=S.gf if bal=="Yes" else S.rf); r+=1
    r+=1; wc(ws,r,1,"Summary",font=S.bf,align=S.lt); r+=1
    nb=sum(1 for _,_,s1,s2 in recs if not np.isnan(s1) and s1<=0.1 and not np.isnan(s2) and s2<=0.1)
    nt=sum(1 for _,_,s1,s2 in recs if not np.isnan(s1) and not np.isnan(s2))
    wc(ws,r,1,f"SMD <= 0.1: {nb}/{nt} ({nb/nt*100:.1f}%)" if nt>0 else "N/A",align=S.lt)
    ws.column_dimensions["A"].width=22; ws.column_dimensions["B"].width=12
    for i in range(3,6): ws.column_dimensions[get_column_letter(i)].width=24
    ws.freeze_panes="A2"


# ============================================================
# Sheet 8: Variable Dictionary
# ============================================================
def write_dict(wb, selected, datasets):
    ws = wb.create_sheet("Variable Dictionary")
    wh(ws,1,["No.","Variable","Label","Type","Group","Value Range"])
    ref = next(iter(datasets.values()),None); r=2
    for i, v in enumerate(selected, 1):
        wc(ws,r,1,i); wc(ws,r,2,v,font=S.bf,align=S.lt)
        wc(ws,r,3,VAR_LABELS.get(v,""),align=S.lt)
        vt=var_type(v); wc(ws,r,4,vt); wc(ws,r,5,var_group(v),align=S.lt)
        rd=""
        if ref is not None and v in ref.columns:
            if vt=="Continuous":
                vv=safe_num(ref[v]).dropna()
                if len(vv)>0: rd=f"Range: {vv.min():.1f} ~ {vv.max():.1f}"
            elif vt=="Binary": rd="True / False"
            else:
                u=ref[v].dropna().unique()
                if v in VALUE_LABELS:
                    labeled = [f"{cat_label(v, x)}" for x in sorted(u, key=lambda x: cat_sort_key(v, x))]
                    rd=", ".join(labeled) if len(labeled)<=10 else f"{len(u)} categories"
                else:
                    rd=", ".join(sorted([str(x) for x in u])) if len(u)<=10 else f"{len(u)} categories"
        wc(ws,r,6,rd,align=S.lt); r+=1
    ws.column_dimensions["A"].width=6; ws.column_dimensions["B"].width=24
    ws.column_dimensions["C"].width=30; ws.column_dimensions["D"].width=12
    ws.column_dimensions["E"].width=24; ws.column_dimensions["F"].width=45
    ws.freeze_panes="A2"


# ============================================================
# Sheet: Outcome-Stratified Table (SA+ vs SA- within one dataset)
# ============================================================
def write_outcome_table(wb, df, ds_name, selected, sgrps):
    """
    Table 1 stratified by outcome (SA+ vs SA-) for a single dataset.
    Continuous: Median(IQR), P from Mann-Whitney U
    Binary:     n(%) positive, P from Chi-square / Fisher
    Multi-cat:  n(%) per level, P from Chi-square
    """
    sheet_name = f"Outcome {ds_name}"
    # Excel sheet name max 31 chars
    if len(sheet_name) > 31:
        sheet_name = sheet_name[:31]
    ws = wb.create_sheet(sheet_name)

    y = outcome_01(df)
    df_pos = df[y == 1]
    df_neg = df[y == 0]
    n_pos = len(df_pos)
    n_neg = len(df_neg)

    hds = ["Group", "Variable", "Label",
           f"SA+ (n={n_pos:,})", f"SA- (n={n_neg:,})", "P-value", "Test"]
    wh(ws, 1, hds)
    r = 2

    for gn, gvs in sgrps.items():
        wc(ws, r, 1, gn, font=S.sft, fill=S.sf, align=S.lt)
        for c in range(2, 8): wc(ws, r, c, "", fill=S.sf)
        r += 1

        for var in gvs:
            if var not in df.columns:
                continue
            lb = VAR_LABELS.get(var, "")

            if var in CONTINUOUS_VARS:
                v_pos = safe_num(df_pos[var]).dropna()
                v_neg = safe_num(df_neg[var]).dropna()
                # Median(IQR)
                def med_iqr(v):
                    if len(v) == 0: return "—"
                    return f"{np.median(v):.2f} ({np.percentile(v,25):.2f}-{np.percentile(v,75):.2f})"
                # Mann-Whitney U
                pu = np.nan
                if len(v_pos) > 0 and len(v_neg) > 0:
                    try: _, pu = stats.mannwhitneyu(v_pos, v_neg, alternative="two-sided")
                    except: pass

                wc(ws, r, 1, ""); wc(ws, r, 2, var, font=S.bf, align=S.lt)
                wc(ws, r, 3, lb, align=S.lt)
                wc(ws, r, 4, med_iqr(v_pos)); wc(ws, r, 5, med_iqr(v_neg))
                pc = wc(ws, r, 6, fmt_p(pu))
                if not np.isnan(pu) and pu < 0.05: pc.font = S.rf
                wc(ws, r, 7, "MWU"); r += 1

            elif var in BINARY_VARS:
                s_pos = df_pos[var].map({True:1,False:0,"TRUE":1,"FALSE":0,"True":1,"False":0,1:1,0:0})
                s_neg = df_neg[var].map({True:1,False:0,"TRUE":1,"FALSE":0,"True":1,"False":0,1:1,0:0})
                a_pos = s_pos.dropna(); a_neg = s_neg.dropna()
                n1 = len(a_pos); n0 = len(a_neg)
                t1 = int(a_pos.sum()) if n1 > 0 else 0
                t0 = int(a_neg.sum()) if n0 > 0 else 0
                p1 = t1/n1*100 if n1 > 0 else 0
                p0 = t0/n0*100 if n0 > 0 else 0

                # Chi-square
                pchi = np.nan
                if n1 > 0 and n0 > 0:
                    try:
                        tab = np.array([[t1, n1-t1], [t0, n0-t0]])
                        if tab.min() < 5:
                            _, pchi = stats.fisher_exact(tab)
                        else:
                            _, pchi, _, _ = stats.chi2_contingency(tab, correction=True)
                    except: pass

                wc(ws, r, 1, ""); wc(ws, r, 2, var, font=S.bf, align=S.lt)
                wc(ws, r, 3, lb, align=S.lt)
                wc(ws, r, 4, f"{t1:,} ({p1:.1f}%)")
                wc(ws, r, 5, f"{t0:,} ({p0:.1f}%)")
                pc = wc(ws, r, 6, fmt_p(pchi))
                if not np.isnan(pchi) and pchi < 0.05: pc.font = S.rf
                test_name = "Fisher" if (n1 > 0 and n0 > 0 and np.array([[t1,n1-t1],[t0,n0-t0]]).min() < 5) else "Chi2"
                wc(ws, r, 7, test_name); r += 1

            elif var in ORDINAL_VARS + NOMINAL_VARS:
                s_pos = df_pos[var].map({True:"True",False:"False",1:"True",0:"False"}).fillna(df_pos[var])
                s_neg = df_neg[var].map({True:"True",False:"False",1:"True",0:"False"}).fillna(df_neg[var])
                vc_pos = s_pos.value_counts(dropna=True)
                vc_neg = s_neg.value_counts(dropna=True)
                n1 = vc_pos.sum(); n0 = vc_neg.sum()
                cats = sorted(set(vc_pos.index) | set(vc_neg.index), key=lambda x: cat_sort_key(var, x))

                pchi = np.nan
                if len(cats) >= 2 and n1 > 0 and n0 > 0:
                    try:
                        tab = pd.DataFrame({"pos": vc_pos.reindex(cats, fill_value=0),
                                            "neg": vc_neg.reindex(cats, fill_value=0)})
                        _, pchi, _, _ = stats.chi2_contingency(tab.values)
                    except: pass

                # Variable header row
                wc(ws, r, 1, ""); wc(ws, r, 2, var, font=S.bf, align=S.lt)
                wc(ws, r, 3, lb, align=S.lt)
                wc(ws, r, 4, ""); wc(ws, r, 5, "")
                pc = wc(ws, r, 6, fmt_p(pchi))
                if not np.isnan(pchi) and pchi < 0.05: pc.font = S.rf
                wc(ws, r, 7, "Chi2"); r += 1

                # Category rows
                for cat in cats:
                    c1 = vc_pos.get(cat, 0); c0 = vc_neg.get(cat, 0)
                    p1 = c1/n1*100 if n1 > 0 else 0
                    p0 = c0/n0*100 if n0 > 0 else 0
                    wc(ws, r, 1, ""); wc(ws, r, 2, "")
                    wc(ws, r, 3, f"  {cat_label(var, cat)}", align=S.lt)
                    wc(ws, r, 4, f"{int(c1):,} ({p1:.1f}%)")
                    wc(ws, r, 5, f"{int(c0):,} ({p0:.1f}%)")
                    wc(ws, r, 6, ""); wc(ws, r, 7, ""); r += 1

    # Note
    r += 1
    note = ("Note: Continuous variables as Median(IQR), P from Mann-Whitney U test; "
            "Binary variables as n(%) positive, P from Chi-square or Fisher's exact test (when expected count <5); "
            "Multi-category variables as n(%) per level, P from Chi-square test. "
            f"Dataset: {ds_name}.")
    wc(ws, r, 1, note, font=S.nt, align=S.lt)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=7)

    ws.column_dimensions["A"].width = 20; ws.column_dimensions["B"].width = 22
    ws.column_dimensions["C"].width = 30; ws.column_dimensions["D"].width = 24
    ws.column_dimensions["E"].width = 24; ws.column_dimensions["F"].width = 12
    ws.column_dimensions["G"].width = 8
    ws.freeze_panes = "D2"


# ============================================================
# Sheet: Combined Outcome-Stratified Summary (all datasets side by side)
# ============================================================
def write_outcome_combined(wb, datasets, selected, sgrps):
    """
    Combined Table 1 stratified by SA+ vs SA- across ALL datasets in one sheet.
    Columns: Group | Variable | Label | DS1 SA+ | DS1 SA- | DS1 P | DS2 SA+ | DS2 SA- | DS2 P | ...
    """
    ws = wb.create_sheet("Outcome Summary")
    dn = list(datasets.keys())

    # Prepare subsets
    subsets = {}
    for nm, df in datasets.items():
        y = outcome_01(df)
        subsets[nm] = {"pos": df[y == 1], "neg": df[y == 0],
                       "n_pos": int(y.sum()), "n_neg": int((y == 0).sum())}

    # Header
    hds = ["Group", "Variable", "Label"]
    for nm in dn:
        sp = subsets[nm]
        hds += [f"{nm}\nSA+ (n={sp['n_pos']:,})",
                f"{nm}\nSA- (n={sp['n_neg']:,})",
                f"{nm}\nP-value"]
    hds.append("Test")
    wh(ws, 1, hds)
    tc_ = len(hds)  # Test column index
    r = 2

    def _med_iqr(v):
        if len(v) == 0: return "—"
        return f"{np.median(v):.2f} ({np.percentile(v,25):.2f}-{np.percentile(v,75):.2f})"

    for gn, gvs in sgrps.items():
        wc(ws, r, 1, gn, font=S.sft, fill=S.sf, align=S.lt)
        for c in range(2, tc_ + 1): wc(ws, r, c, "", fill=S.sf)
        r += 1

        for var in gvs:
            if not any(var in datasets[nm].columns for nm in dn):
                continue
            lb = VAR_LABELS.get(var, "")

            if var in CONTINUOUS_VARS:
                wc(ws, r, 1, ""); wc(ws, r, 2, var, font=S.bf, align=S.lt)
                wc(ws, r, 3, lb, align=S.lt)
                col = 4
                for nm in dn:
                    if var in datasets[nm].columns:
                        vp = safe_num(subsets[nm]["pos"][var]).dropna()
                        vn = safe_num(subsets[nm]["neg"][var]).dropna()
                        wc(ws, r, col, _med_iqr(vp)); wc(ws, r, col + 1, _med_iqr(vn))
                        pu = np.nan
                        if len(vp) > 0 and len(vn) > 0:
                            try: _, pu = stats.mannwhitneyu(vp, vn, alternative="two-sided")
                            except: pass
                        pc = wc(ws, r, col + 2, fmt_p(pu))
                        if not np.isnan(pu) and pu < 0.05: pc.font = S.rf
                    else:
                        wc(ws, r, col, "—"); wc(ws, r, col + 1, "—"); wc(ws, r, col + 2, "")
                    col += 3
                wc(ws, r, tc_, "MWU"); r += 1

            elif var in BINARY_VARS:
                wc(ws, r, 1, ""); wc(ws, r, 2, var, font=S.bf, align=S.lt)
                wc(ws, r, 3, lb, align=S.lt)
                col = 4
                for nm in dn:
                    if var in datasets[nm].columns:
                        sp = subsets[nm]
                        mp = {True: 1, False: 0, "TRUE": 1, "FALSE": 0, "True": 1, "False": 0, 1: 1, 0: 0}
                        ap = sp["pos"][var].map(mp).dropna(); an = sp["neg"][var].map(mp).dropna()
                        n1 = len(ap); n0 = len(an)
                        t1 = int(ap.sum()) if n1 > 0 else 0
                        t0 = int(an.sum()) if n0 > 0 else 0
                        wc(ws, r, col, f"{t1:,} ({t1/n1*100:.1f}%)" if n1 > 0 else "—")
                        wc(ws, r, col + 1, f"{t0:,} ({t0/n0*100:.1f}%)" if n0 > 0 else "—")
                        pchi = np.nan
                        if n1 > 0 and n0 > 0:
                            try:
                                tab = np.array([[t1, n1 - t1], [t0, n0 - t0]])
                                if tab.min() < 5:
                                    _, pchi = stats.fisher_exact(tab)
                                else:
                                    _, pchi, _, _ = stats.chi2_contingency(tab, correction=True)
                            except: pass
                        pc = wc(ws, r, col + 2, fmt_p(pchi))
                        if not np.isnan(pchi) and pchi < 0.05: pc.font = S.rf
                    else:
                        wc(ws, r, col, "—"); wc(ws, r, col + 1, "—"); wc(ws, r, col + 2, "")
                    col += 3
                wc(ws, r, tc_, "Chi2/Fisher"); r += 1

            elif var in ORDINAL_VARS + NOMINAL_VARS:
                # Variable header row
                wc(ws, r, 1, ""); wc(ws, r, 2, var, font=S.bf, align=S.lt)
                wc(ws, r, 3, lb, align=S.lt)
                col = 4
                # Collect all categories across all datasets
                all_cats = set()
                vc_data = {}
                for nm in dn:
                    if var not in datasets[nm].columns: continue
                    sp = subsets[nm]
                    mp = {True: "True", False: "False", 1: "True", 0: "False"}
                    vp = sp["pos"][var].map(mp).fillna(sp["pos"][var]).value_counts(dropna=True)
                    vn = sp["neg"][var].map(mp).fillna(sp["neg"][var]).value_counts(dropna=True)
                    all_cats.update(vp.index); all_cats.update(vn.index)
                    vc_data[nm] = {"vp": vp, "np": vp.sum(), "vn": vn, "nn": vn.sum()}
                all_cats = sorted(all_cats, key=lambda x: cat_sort_key(var, x))

                # P-values per dataset on the header row
                for nm in dn:
                    if nm in vc_data and len(all_cats) >= 2:
                        d = vc_data[nm]
                        wc(ws, r, col, ""); wc(ws, r, col + 1, "")
                        pchi = np.nan
                        try:
                            tab = pd.DataFrame({
                                "p": pd.Series(d["vp"]).reindex(all_cats, fill_value=0),
                                "n": pd.Series(d["vn"]).reindex(all_cats, fill_value=0)})
                            _, pchi, _, _ = stats.chi2_contingency(tab.values)
                        except: pass
                        pc = wc(ws, r, col + 2, fmt_p(pchi))
                        if not np.isnan(pchi) and pchi < 0.05: pc.font = S.rf
                    else:
                        wc(ws, r, col, ""); wc(ws, r, col + 1, ""); wc(ws, r, col + 2, "")
                    col += 3
                wc(ws, r, tc_, "Chi2"); r += 1

                # Category rows
                for cat in all_cats:
                    wc(ws, r, 1, ""); wc(ws, r, 2, "")
                    wc(ws, r, 3, f"  {cat_label(var, cat)}", align=S.lt)
                    col = 4
                    for nm in dn:
                        if nm in vc_data:
                            d = vc_data[nm]
                            c1 = d["vp"].get(cat, 0); c0 = d["vn"].get(cat, 0)
                            wc(ws, r, col, f"{int(c1):,} ({c1/d['np']*100:.1f}%)" if d["np"] > 0 else "—")
                            wc(ws, r, col + 1, f"{int(c0):,} ({c0/d['nn']*100:.1f}%)" if d["nn"] > 0 else "—")
                            wc(ws, r, col + 2, "")
                        else:
                            wc(ws, r, col, "—"); wc(ws, r, col + 1, "—"); wc(ws, r, col + 2, "")
                        col += 3
                    wc(ws, r, tc_, ""); r += 1

    # Note
    r += 1
    note = ("Note: Continuous variables as Median(IQR), P from Mann-Whitney U; "
            "Binary variables as n(%) positive, P from Chi-square or Fisher's exact (expected <5); "
            "Multi-category as n(%) per level, P from Chi-square. "
            "SA+ = Spontaneous Abortion positive; SA- = negative.")
    wc(ws, r, 1, note, font=S.nt, align=S.lt)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=tc_)

    # Column widths
    ws.column_dimensions["A"].width = 18
    ws.column_dimensions["B"].width = 20
    ws.column_dimensions["C"].width = 28
    col_idx = 4
    for nm in dn:
        ws.column_dimensions[get_column_letter(col_idx)].width = 24      # SA+
        ws.column_dimensions[get_column_letter(col_idx + 1)].width = 24  # SA-
        ws.column_dimensions[get_column_letter(col_idx + 2)].width = 10  # P
        col_idx += 3
    ws.column_dimensions[get_column_letter(tc_)].width = 12
    ws.freeze_panes = "D2"


# ============================================================
# Main
# ============================================================
def create_report(datasets, selected, boruta_df, output):
    sc, so, sn, sb = classify(selected); sg = sel_groups(selected)
    wb = Workbook()
    print("\nGenerating report...")
    print(f"  Selected: {len(selected)} features (Cont:{len(sc)} Ord:{len(so)} Nom:{len(sn)} Bin:{len(sb)})")
    print("  Sheet 1:  Feature Selection Summary"); write_selection_summary(wb, selected, sc, so, sn, sb, sg)
    if boruta_df is not None:
        print("  Sheet 2:  Boruta Importance"); write_boruta_importance(wb, boruta_df, set(selected))
    print("  Sheet 3:  Dataset Overview"); write_overview(wb, datasets, selected)
    print("  Sheet 4:  Table 1 (Selected)"); write_table1(wb, datasets, selected, sg)
    if sc: print("  Sheet 5:  Continuous Variables"); write_continuous(wb, datasets, sc)
    if so or sn or sb: print("  Sheet 6:  Categorical Variables"); write_categorical(wb, datasets, so, sn, sb)
    print("  Sheet 7:  SMD Summary"); write_smd(wb, datasets, sc, sb)
    print("  Sheet 8:  Variable Dictionary"); write_dict(wb, selected, datasets)
    # Outcome-stratified tables for each dataset
    sheet_idx = 9
    for ds_name, df in datasets.items():
        if OUTCOME_VAR in df.columns:
            print(f"  Sheet {sheet_idx}: Outcome-Stratified ({ds_name})")
            write_outcome_table(wb, df, ds_name, selected, sg)
            sheet_idx += 1
    # Combined outcome-stratified summary
    print(f"  Sheet {sheet_idx}: Outcome Summary (all datasets)")
    write_outcome_combined(wb, datasets, selected, sg)
    wb.save(output); print(f"\nReport saved: {output}")


# ============================================================
# Entry Point
# ============================================================
if __name__ == "__main__":

    # ---- Read selected features ----
    selected = []
    if os.path.exists(SELECTED_FEATURES_PATH):
        print(f"Reading selected features: {SELECTED_FEATURES_PATH}")
        df_sel = pd.read_csv(SELECTED_FEATURES_PATH)
        col_map = {c.lower().strip(): c for c in df_sel.columns}
        var_col = None
        for k in ["variable", "feature", "var", "name"]:
            if k in col_map: var_col = col_map[k]; break
        if var_col is None: var_col = df_sel.columns[0]
        selected = df_sel[var_col].astype(str).str.strip().tolist()
        selected = [v for v in selected if v not in {OUTCOME_VAR, INDEX_VAR, DATE_VAR, "Dataset"}]
        print(f"  {len(selected)} features loaded")
    else:
        print(f"Warning: {SELECTED_FEATURES_PATH} not found")

    # ---- Read Boruta full results ----
    boruta_df = None
    if os.path.exists(BORUTA_RESULTS_PATH):
        print(f"Reading Boruta results: {BORUTA_RESULTS_PATH}")
        boruta_df = pd.read_csv(BORUTA_RESULTS_PATH)
        print(f"  {len(boruta_df)} variables with importance metrics")
    else:
        print(f"Warning: {BORUTA_RESULTS_PATH} not found (Boruta Importance sheet will be skipped)")

    # If no selected_features.csv, try to extract from boruta_results.csv
    if not selected and boruta_df is not None:
        print("  Extracting confirmed features from boruta_results.csv ...")
        col_map = {c.lower().strip(): c for c in boruta_df.columns}
        dec_col = col_map.get("decision", None)
        var_col = None
        for k in ["variable", "feature", "var", "name"]:
            if k in col_map: var_col = col_map[k]; break
        if var_col is None: var_col = boruta_df.columns[0]
        if dec_col:
            confirmed = boruta_df[boruta_df[dec_col].astype(str).str.strip().str.lower().isin(
                ["confirmed","accepted","selected","true","1","yes"])]
            selected = confirmed[var_col].astype(str).str.strip().tolist()
            selected = [v for v in selected if v not in {OUTCOME_VAR, INDEX_VAR, DATE_VAR}]
        print(f"  {len(selected)} confirmed features extracted")

    if not selected:
        print("\nError: No selected features found!"); sys.exit(1)

    # ---- Read datasets ----
    print("\nReading datasets...")
    datasets = {}
    for path, name in [(TRAIN_PATH, DS_TRAIN), (INTVAL_PATH, DS_INTVAL), (TEMPVAL_PATH, DS_TEMPVAL)]:
        if os.path.exists(path):
            df = pd.read_csv(path); datasets[name] = df
            print(f"  {name}: {len(df):,} rows, {len(df.columns)} cols")
        else:
            print(f"  Warning: {path} not found")

    if not datasets:
        print("\nError: No datasets found!"); sys.exit(1)

    # Validate
    ref_cols = set(next(iter(datasets.values())).columns)
    valid = [v for v in selected if v in ref_cols]
    if len(valid) < len(selected):
        dropped = set(selected) - set(valid)
        print(f"  Warning: {len(dropped)} features not in data columns: {dropped}")
    selected = valid
    if not selected:
        print("\nError: No selected features found in datasets!"); sys.exit(1)

    # ---- Generate ----
    print(f"\n{'='*80}")
    print("POST-FEATURE-SELECTION DESCRIPTIVE STATISTICS")
    print(f"{'='*80}")
    dn = list(datasets.keys())
    print(f"\nFeatures: {len(selected)} / {len(ALL_VARS)} ({len(selected)/len(ALL_VARS)*100:.1f}%)")
    for n in dn:
        y = outcome_01(datasets[n])
        print(f"  {n:28s}: N={len(datasets[n]):>7,}  SA={int(y.sum()):>5,} ({y.mean()*100:.2f}%)")

    os.makedirs(os.path.dirname(OUTPUT_PATH) if os.path.dirname(OUTPUT_PATH) else ".", exist_ok=True)
    create_report(datasets, selected, boruta_df, OUTPUT_PATH)

    print(f"\nDone! Report: {OUTPUT_PATH}")
    print("  Sheets: Selection Summary / Boruta Importance / Overview / Table 1 /")
    print("    Continuous / Categorical / SMD Summary / Variable Dictionary /")
    print("    Outcome Training / Outcome Internal Val / Outcome Temporal Val /")
    print("    Outcome Summary (combined)")

Reading selected features: ./Data\selected_features.csv
  36 features loaded
Reading Boruta results: ./Data\boruta_results.csv
  134 variables with importance metrics

Reading datasets...
  Training: 360,363 rows, 39 cols
  Internal Validation: 40,041 rows, 39 cols
  Temporal Validation: 1,822 rows, 39 cols

POST-FEATURE-SELECTION DESCRIPTIVE STATISTICS

Features: 36 / 134 (26.9%)
  Training                    : N=360,363  SA=11,753 (3.26%)
  Internal Validation         : N= 40,041  SA=1,306 (3.26%)
  Temporal Validation         : N=  1,822  SA=  262 (14.38%)

Generating report...
  Selected: 36 features (Cont:22 Ord:4 Nom:3 Bin:7)
  Sheet 1:  Feature Selection Summary
  Sheet 2:  Boruta Importance
  Sheet 3:  Dataset Overview
  Sheet 4:  Table 1 (Selected)
  Sheet 5:  Continuous Variables
  Sheet 6:  Categorical Variables
  Sheet 7:  SMD Summary
  Sheet 8:  Variable Dictionary
  Sheet 9: Outcome-Stratified (Training)
  Sheet 10: Outcome-Stratified (Internal Validation)
  Sheet 11: Out

In [17]:
"""
Feature Correlation Analysis (Post-Feature-Selection)
=======================================================
Comprehensive correlation analysis for Boruta-selected features on the training set.

Expected Input Files (in ./Data/):
  selected_features.csv                      List of confirmed features
  data_training_selected.csv                 Training set (selected features only)

Output:
  ./Data/correlation_analysis.xlsx           Excel report with correlation matrices & VIF
  ./Data/correlation_heatmap.png             Heatmap of all selected features
  ./Data/correlation_heatmap_continuous.png  Heatmap of continuous features only
  ./Data/high_correlation_pairs.csv          Pairs with |r| > threshold

Excel Report Sheets:
  1. Correlation Matrix (All)      — Spearman correlation matrix for all selected features
  2. Correlation Matrix (Cont)     — Pearson/Spearman for continuous features only
  3. High Correlation Pairs        — Feature pairs with |r| > threshold, sorted by |r|
  4. VIF Analysis                  — Variance Inflation Factor for continuous features
  5. Feature-Outcome Correlation   — Each feature's association with outcome (point-biserial / rank-biserial)
  6. Correlation Summary           — Overview statistics and recommendations

Dependencies:
  pip install pandas numpy scipy matplotlib seaborn openpyxl statsmodels scikit-learn
"""

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings
import sys
import os

warnings.filterwarnings("ignore")

# ============================================================
# Configuration
# ============================================================
OUTCOME_VAR = "SpontAbortion"
INDEX_VAR = "Index"
DATE_VAR = "BaseInfoDate"

DATA_DIR = "./Data"
SELECTED_FEATURES_PATH = os.path.join(DATA_DIR, "selected_features.csv")
TRAIN_PATH = os.path.join(DATA_DIR, "data_training_selected.csv")

OUTPUT_XLSX = os.path.join(DATA_DIR, "correlation_analysis.xlsx")
OUTPUT_HEATMAP_ALL = os.path.join(DATA_DIR, "correlation_heatmap.png")
OUTPUT_HEATMAP_CONT = os.path.join(DATA_DIR, "correlation_heatmap_continuous.png")
OUTPUT_HIGH_CORR = os.path.join(DATA_DIR, "high_correlation_pairs.csv")

HIGH_CORR_THRESHOLD = 0.7   # |r| threshold for flagging high correlation
VIF_THRESHOLD = 10.0         # VIF threshold for flagging multicollinearity

# ============================================================
# Variable Definitions
# ============================================================
CONTINUOUS_VARS = [
    "WifeAge", "HusbandAge", "MenarcheAge",
    "WifeHeight", "WifeBMI", "WifeHR", "WifeSBP", "WifeDBP",
    "HusbHeight", "HusbBMI", "HusbHR", "HusbSBP", "HusbDBP",
    "Hemoglobin", "RBC", "Platelet", "WBC",
    "NeutrophilPct", "LymphocytePct",
    "Glucose", "WifeALT", "WifeCreat", "TSH",
    "HusbALT", "HusbCreat",
    "LeftTestVol", "RightTestVol",
]
ORDINAL_VARS = ["WifeEdu", "HusbandEdu", "WifeOcc", "HusbandOcc"]
NOMINAL_VARS = [
    "WifeEthnic", "HusbandEthnic", "WifeResident", "HusbandResident",
    "WifeABO", "HusbABO", "CycleRegular", "MenstrualVol", "Dysmenorrhea",
    "WifeMental", "WifeIntel", "WifeFace", "WifePosture", "WifeFaceSpec", "WifeSkinHair",
    "HusbMental", "HusbIntel", "HusbFace", "HusbPosture", "HusbFaceSpec", "HusbSkinHair",
    "WifePubHair", "Breast", "Vulva",
    "HusbPubHair", "AdamsApple", "Penis", "Foreskin",
    "Testis", "Epididymis", "VasDeferens", "Varicocele",
    "WifeUrine", "HusbUrine", "WifeHepB", "HusbHepB",
]
BINARY_VARS = [
    "WifeDiseaseHx", "WifeBirthDefect", "GynDisease",
    "WifeMedication", "WifeVaccine", "Contraception",
    "PrevPregnancy", "Consanguinity", "WifeFamConsang", "WifeFamDisease",
    "WifeMeatEgg", "WifeVegAverse", "WifeRawMeat",
    "WifeSmoke", "WifePassSmoke", "WifeAlcohol",
    "WifeDrugUse", "WifeHalitosis", "WifeGumBleed",
    "WifeEnvExpose", "WifeStress", "WifeRelatStress", "WifeFinance", "WifeReady",
    "WifeThyroid", "WifeLung", "WifeRhythm", "WifeMurmur",
    "WifeLiverSpleen", "WifeExtremity", "WifeGynExam",
    "WifeRh", "HusbRh",
    "RubellaIgG", "CMVIgG", "CMVIgM", "ToxoIgG", "ToxoIgM",
    "SyphilisScr", "HusbSyphilis", "GynUltrasound",
    "HusbDiseaseHis", "HusbBirthDefect", "HusbAndrologyDis",
    "HusbMedication", "HusbVaccinated",
    "HusbFamConsang", "HusbFamDisease",
    "HusbMeatEgg", "HusbVegAverse", "HusbRawMeat",
    "HusbSmoke", "HusbPassSmoke", "HusbAlcohol", "HusbDrugUse",
    "HusbEnvExpose", "HusbStress", "HusbRelatStress", "HusbFinance", "HusbReady",
    "HusbThyroid", "HusbLung", "HusbRhythm", "HusbMurmur",
    "HusbLiverSpleen", "HusbExtremity", "HusbAndroExam",
]

VAR_LABELS = {
    "SpontAbortion": "Spontaneous Abortion",
    "WifeAge": "Wife Age", "HusbandAge": "Husband Age", "MenarcheAge": "Menarche Age",
    "WifeHeight": "Wife Height", "WifeBMI": "Wife BMI",
    "WifeHR": "Wife HR", "WifeSBP": "Wife SBP", "WifeDBP": "Wife DBP",
    "HusbHeight": "Husband Height", "HusbBMI": "Husband BMI",
    "HusbHR": "Husband HR", "HusbSBP": "Husband SBP", "HusbDBP": "Husband DBP",
    "Hemoglobin": "Hemoglobin", "RBC": "RBC", "Platelet": "Platelet", "WBC": "WBC",
    "NeutrophilPct": "Neutrophil%", "LymphocytePct": "Lymphocyte%",
    "Glucose": "Glucose", "WifeALT": "Wife ALT", "WifeCreat": "Wife Creatinine",
    "TSH": "TSH", "HusbALT": "Husband ALT", "HusbCreat": "Husband Creatinine",
    "LeftTestVol": "Left Test Vol", "RightTestVol": "Right Test Vol",
    "WifeEdu": "Wife Education", "HusbandEdu": "Husband Education",
    "WifeOcc": "Wife Occupation", "HusbandOcc": "Husband Occupation",
}

def var_type(v):
    if v in CONTINUOUS_VARS: return "Continuous"
    if v in ORDINAL_VARS: return "Ordinal"
    if v in NOMINAL_VARS: return "Nominal"
    if v in BINARY_VARS: return "Binary"
    return "Unknown"


# ============================================================
# Excel Styles
# ============================================================
class S:
    hf = Font(bold=True, size=11, color="FFFFFF", name="Arial")
    hfl = PatternFill("solid", fgColor="4472C4")
    sf = PatternFill("solid", fgColor="E2EFDA")
    sft = Font(bold=True, size=11, name="Arial", color="375623")
    nf = Font(size=10, name="Arial")
    bf = Font(size=10, name="Arial", bold=True)
    rf = Font(size=10, name="Arial", bold=True, color="C00000")
    gf = Font(size=10, name="Arial", bold=True, color="006100")
    nt = Font(size=9, name="Arial", italic=True, color="666666")
    bd = Border(left=Side(style="thin", color="BFBFBF"), right=Side(style="thin", color="BFBFBF"),
                top=Side(style="thin", color="BFBFBF"), bottom=Side(style="thin", color="BFBFBF"))
    ct = Alignment(horizontal="center")
    wc = Alignment(horizontal="center", wrap_text=True)
    lt = Alignment(horizontal="left")

def wh(ws, r, hds):
    for c, h in enumerate(hds, 1):
        cl = ws.cell(row=r, column=c, value=h)
        cl.font = S.hf; cl.fill = S.hfl; cl.alignment = S.wc; cl.border = S.bd

def wcl(ws, r, c, v, font=None, fill=None, align=None):
    cl = ws.cell(row=r, column=c, value=v)
    cl.font = font or S.nf; cl.border = S.bd; cl.alignment = align or S.ct
    if fill: cl.fill = fill
    return cl


# ============================================================
# Correlation color helper
# ============================================================
def corr_fill(val, threshold=HIGH_CORR_THRESHOLD):
    """Return fill color based on correlation magnitude."""
    if pd.isna(val): return None
    av = abs(val)
    if av >= threshold:
        return PatternFill("solid", fgColor="FFC7CE")  # red
    elif av >= 0.5:
        return PatternFill("solid", fgColor="FFEB9C")  # yellow
    elif av >= 0.3:
        return PatternFill("solid", fgColor="F2F2F2")  # light gray
    return None


# ============================================================
# Prepare numeric data for correlation
# ============================================================
def prepare_numeric(df, selected):
    """
    Convert selected features to numeric for correlation analysis.
    - Continuous: as-is
    - Binary: map True/False to 1/0
    - Ordinal/Nominal: encode to numeric (ordinal keeps order, nominal uses label encoding)
    """
    df_num = pd.DataFrame(index=df.index)

    for var in selected:
        if var not in df.columns:
            continue

        if var in CONTINUOUS_VARS:
            df_num[var] = pd.to_numeric(df[var], errors="coerce")

        elif var in BINARY_VARS:
            df_num[var] = df[var].map(
                {True: 1, False: 0, "TRUE": 1, "FALSE": 0, "True": 1, "False": 0, 1: 1, 0: 0}
            ).astype(float)

        elif var in ORDINAL_VARS:
            # Ordinal: map to integers preserving order
            vals = df[var].dropna().unique()
            vals_sorted = sorted(vals, key=str)
            mapping = {v: i for i, v in enumerate(vals_sorted)}
            df_num[var] = df[var].map(mapping).astype(float)

        elif var in NOMINAL_VARS:
            # Nominal: label encode
            vals = df[var].dropna().unique()
            mapping = {v: i for i, v in enumerate(sorted(vals, key=str))}
            df_num[var] = df[var].map(mapping).astype(float)

    return df_num


# ============================================================
# Sheet 1 & 2: Correlation Matrices
# ============================================================
def write_corr_matrix(wb, corr_df, sheet_name, p_matrix=None):
    ws = wb.create_sheet(sheet_name)
    feats = corr_df.columns.tolist()
    n = len(feats)

    # Header row
    wh(ws, 1, [""] + feats)
    # Row header column
    for i, f in enumerate(feats):
        wcl(ws, i + 2, 1, f, font=S.bf, align=S.lt)

    # Fill matrix
    for i in range(n):
        for j in range(n):
            val = corr_df.iloc[i, j]
            if pd.notna(val):
                cell = wcl(ws, i + 2, j + 2, round(val, 3))
                fill = corr_fill(val)
                if fill: cell.fill = fill
                if i == j:
                    cell.fill = PatternFill("solid", fgColor="D6E4F0")
                # Bold if significant (p < 0.05) and off-diagonal
                if p_matrix is not None and i != j:
                    pv = p_matrix.iloc[i, j]
                    if pd.notna(pv) and pv < 0.05 and abs(val) >= HIGH_CORR_THRESHOLD:
                        cell.font = S.rf
            else:
                wcl(ws, i + 2, j + 2, "")

    ws.column_dimensions["A"].width = 20
    for i in range(n):
        ws.column_dimensions[get_column_letter(i + 2)].width = 12
    ws.freeze_panes = "B2"


# ============================================================
# Sheet 3: High Correlation Pairs
# ============================================================
def write_high_corr_pairs(wb, corr_df, p_matrix, threshold):
    ws = wb.create_sheet("High Correlation Pairs")
    feats = corr_df.columns.tolist()
    n = len(feats)

    wh(ws, 1, ["Rank", "Variable 1", "Type 1", "Variable 2", "Type 2",
                "Correlation (r)", "P-value", "|r|", "Concern Level"])

    pairs = []
    for i in range(n):
        for j in range(i + 1, n):
            r_val = corr_df.iloc[i, j]
            p_val = p_matrix.iloc[i, j] if p_matrix is not None else np.nan
            if pd.notna(r_val):
                pairs.append((feats[i], feats[j], r_val, p_val, abs(r_val)))

    pairs.sort(key=lambda x: x[4], reverse=True)

    r = 2
    for rank, (v1, v2, rval, pval, absval) in enumerate(pairs, 1):
        if absval < 0.3 and rank > 100:
            break  # Skip low-correlation pairs beyond rank 100

        concern = ""
        if absval >= 0.9: concern = "Very High"
        elif absval >= HIGH_CORR_THRESHOLD: concern = "High"
        elif absval >= 0.5: concern = "Moderate"
        elif absval >= 0.3: concern = "Low"
        else: concern = "Negligible"

        wcl(ws, r, 1, rank)
        wcl(ws, r, 2, v1, font=S.bf, align=S.lt)
        wcl(ws, r, 3, var_type(v1))
        wcl(ws, r, 4, v2, font=S.bf, align=S.lt)
        wcl(ws, r, 5, var_type(v2))
        c = wcl(ws, r, 6, round(rval, 4))
        if absval >= HIGH_CORR_THRESHOLD: c.font = S.rf
        wcl(ws, r, 7, f"{pval:.2e}" if pd.notna(pval) and pval > 0 else "<1e-300" if pd.notna(pval) else "")
        wcl(ws, r, 8, round(absval, 4))

        cf = S.rf if concern in ("Very High", "High") else S.gf if concern == "Negligible" else S.nf
        wcl(ws, r, 9, concern, font=cf)
        r += 1

    # Note
    r += 1
    wcl(ws, r, 1, f"Note: Threshold for high correlation: |r| >= {threshold}. "
        "Very High >= 0.9, High >= 0.7, Moderate >= 0.5, Low >= 0.3.",
        font=S.nt, align=S.lt)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=9)

    ws.column_dimensions["A"].width = 6; ws.column_dimensions["B"].width = 20
    ws.column_dimensions["C"].width = 12; ws.column_dimensions["D"].width = 20
    ws.column_dimensions["E"].width = 12; ws.column_dimensions["F"].width = 16
    ws.column_dimensions["G"].width = 14; ws.column_dimensions["H"].width = 10
    ws.column_dimensions["I"].width = 14
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = f"A1:I{r - 2}"

    return pairs


# ============================================================
# Sheet 4: VIF Analysis
# ============================================================
def write_vif(wb, df_num, sel_cont):
    ws = wb.create_sheet("VIF Analysis")
    wh(ws, 1, ["Variable", "Label", "VIF", "Concern"])

    cont_in_data = [v for v in sel_cont if v in df_num.columns]
    if len(cont_in_data) < 2:
        wcl(ws, 2, 1, "Not enough continuous variables for VIF analysis.", align=S.lt)
        return

    from statsmodels.stats.outliers_influence import variance_inflation_factor

    X = df_num[cont_in_data].dropna()
    if len(X) < 10:
        wcl(ws, 2, 1, "Not enough complete observations for VIF.", align=S.lt)
        return

    # Standardize to avoid numerical issues
    X_std = (X - X.mean()) / X.std()
    X_std = X_std.dropna(axis=1)

    vif_data = []
    for i, col in enumerate(X_std.columns):
        try:
            vif = variance_inflation_factor(X_std.values, i)
        except Exception:
            vif = np.nan
        vif_data.append((col, vif))

    # Sort by VIF descending
    vif_data.sort(key=lambda x: x[1] if not np.isnan(x[1]) else 0, reverse=True)

    r = 2
    for var, vif in vif_data:
        wcl(ws, r, 1, var, font=S.bf, align=S.lt)
        wcl(ws, r, 2, VAR_LABELS.get(var, ""), align=S.lt)
        c = wcl(ws, r, 3, round(vif, 2) if not np.isnan(vif) else "—")
        if not np.isnan(vif) and vif >= VIF_THRESHOLD:
            c.font = S.rf
            concern = "High Multicollinearity"
        elif not np.isnan(vif) and vif >= 5:
            concern = "Moderate"
        elif not np.isnan(vif):
            concern = "OK"
        else:
            concern = "—"
        cf = S.rf if "High" in concern else S.gf if concern == "OK" else S.nf
        wcl(ws, r, 4, concern, font=cf)
        r += 1

    r += 1
    wcl(ws, r, 1, f"Note: VIF >= {VIF_THRESHOLD} indicates high multicollinearity. "
        "VIF >= 5 is moderate. Computed on standardized continuous features.",
        font=S.nt, align=S.lt)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=4)

    ws.column_dimensions["A"].width = 22; ws.column_dimensions["B"].width = 24
    ws.column_dimensions["C"].width = 12; ws.column_dimensions["D"].width = 22
    ws.freeze_panes = "A2"


# ============================================================
# Sheet 5: Feature-Outcome Correlation
# ============================================================
def write_outcome_corr(wb, df, df_num, selected):
    ws = wb.create_sheet("Feature-Outcome Corr")
    wh(ws, 1, ["Rank", "Variable", "Label", "Type", "Method",
                "Correlation", "P-value", "|r|", "Direction"])

    y = df[OUTCOME_VAR].map(
        {True: 1, False: 0, "TRUE": 1, "FALSE": 0, "True": 1, "False": 0, 1: 1, 0: 0}
    ).astype(float)

    results = []
    for var in selected:
        if var not in df_num.columns:
            continue

        x = df_num[var]
        valid = x.notna() & y.notna()
        xv = x[valid]; yv = y[valid]

        if len(xv) < 10:
            continue

        vt = var_type(var)
        if vt == "Continuous":
            # Point-biserial (equivalent to Pearson with binary outcome)
            r_val, p_val = stats.pointbiserialr(yv, xv)
            method = "Point-Biserial"
        elif vt == "Binary":
            # Phi coefficient (Pearson on two binary vars)
            r_val, p_val = stats.pearsonr(xv, yv)
            method = "Phi Coefficient"
        else:
            # Spearman for ordinal/nominal
            r_val, p_val = stats.spearmanr(xv, yv)
            method = "Spearman"

        direction = "Positive" if r_val > 0 else "Negative" if r_val < 0 else "None"
        results.append((var, vt, method, r_val, p_val, abs(r_val), direction))

    results.sort(key=lambda x: x[5], reverse=True)

    r = 2
    for rank, (var, vt, method, rval, pval, absval, direction) in enumerate(results, 1):
        wcl(ws, r, 1, rank)
        wcl(ws, r, 2, var, font=S.bf, align=S.lt)
        wcl(ws, r, 3, VAR_LABELS.get(var, ""), align=S.lt)
        wcl(ws, r, 4, vt)
        wcl(ws, r, 5, method)
        c = wcl(ws, r, 6, round(rval, 4))
        if pval < 0.05: c.font = S.rf if absval >= 0.1 else S.bf
        wcl(ws, r, 7, f"{pval:.2e}" if pval > 0 else "<1e-300")
        wcl(ws, r, 8, round(absval, 4))
        wcl(ws, r, 9, direction)
        r += 1

    r += 1
    wcl(ws, r, 1, "Note: Point-biserial for continuous vs binary outcome; "
        "Phi coefficient for binary vs binary; Spearman for ordinal/nominal. "
        "Sorted by |r| descending.", font=S.nt, align=S.lt)
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=9)

    ws.column_dimensions["A"].width = 6; ws.column_dimensions["B"].width = 22
    ws.column_dimensions["C"].width = 20; ws.column_dimensions["D"].width = 12
    ws.column_dimensions["E"].width = 16; ws.column_dimensions["F"].width = 14
    ws.column_dimensions["G"].width = 14; ws.column_dimensions["H"].width = 10
    ws.column_dimensions["I"].width = 12
    ws.freeze_panes = "A2"


# ============================================================
# Sheet 6: Summary
# ============================================================
def write_summary(wb, corr_all, pairs, n_features, n_cont, vif_max=None):
    ws = wb.create_sheet("Correlation Summary")

    wh(ws, 1, ["Metric", "Value"])
    rows = [
        ("Total Selected Features", n_features),
        ("Continuous Features", n_cont),
        ("Correlation Method (All)", "Spearman"),
        ("Correlation Method (Continuous)", "Spearman"),
        (f"High Correlation Threshold", f"|r| >= {HIGH_CORR_THRESHOLD}"),
        (f"VIF Threshold", f">= {VIF_THRESHOLD}"),
        ("", ""),
        ("Total Feature Pairs", len(pairs)),
        ("Pairs |r| >= 0.9", sum(1 for _, _, _, _, a in pairs if a >= 0.9)),
        (f"Pairs |r| >= {HIGH_CORR_THRESHOLD}", sum(1 for _, _, _, _, a in pairs if a >= HIGH_CORR_THRESHOLD)),
        ("Pairs |r| >= 0.5", sum(1 for _, _, _, _, a in pairs if a >= 0.5)),
        ("Pairs |r| >= 0.3", sum(1 for _, _, _, _, a in pairs if a >= 0.3)),
        ("", ""),
        ("Mean |r| (all pairs)", f"{np.mean([a for _,_,_,_,a in pairs]):.4f}" if pairs else "—"),
        ("Median |r| (all pairs)", f"{np.median([a for _,_,_,_,a in pairs]):.4f}" if pairs else "—"),
        ("Max |r| (off-diagonal)", f"{max([a for _,_,_,_,a in pairs]):.4f}" if pairs else "—"),
    ]

    if vif_max is not None:
        rows.append(("Max VIF", f"{vif_max:.2f}"))

    r = 2
    for label, val in rows:
        if label == "":
            r += 1; continue
        wcl(ws, r, 1, label, font=S.bf, align=S.lt)
        wcl(ws, r, 2, str(val), align=S.lt)
        r += 1

    # Recommendations
    r += 1
    wcl(ws, r, 1, "Recommendations", font=S.sft, fill=S.sf, align=S.lt)
    wcl(ws, r, 2, "", fill=S.sf); r += 1

    high_pairs = [(v1, v2, rv) for v1, v2, rv, _, a in pairs if a >= HIGH_CORR_THRESHOLD]
    if high_pairs:
        wcl(ws, r, 1, f"Warning: {len(high_pairs)} feature pair(s) with |r| >= {HIGH_CORR_THRESHOLD}:",
            font=S.rf, align=S.lt); r += 1
        for v1, v2, rv in high_pairs[:10]:
            wcl(ws, r, 1, f"  {v1} <-> {v2}: r = {rv:.3f}", align=S.lt); r += 1
        if len(high_pairs) > 10:
            wcl(ws, r, 1, f"  ... and {len(high_pairs)-10} more (see High Correlation Pairs sheet)", align=S.lt); r += 1
        r += 1
        wcl(ws, r, 1, "Consider: removing one feature from each highly correlated pair, "
            "or using regularization (L1/L2) in downstream modeling.", font=S.nt, align=S.lt); r += 1
    else:
        wcl(ws, r, 1, f"No feature pairs with |r| >= {HIGH_CORR_THRESHOLD}. "
            "No severe multicollinearity concerns.", font=S.gf, align=S.lt); r += 1

    ws.column_dimensions["A"].width = 60; ws.column_dimensions["B"].width = 30
    ws.freeze_panes = "A2"


# ============================================================
# Heatmap Generation
# ============================================================
def plot_heatmap(corr_df, output_path, title, figsize=None):
    n = len(corr_df)
    if figsize is None:
        figsize = (max(10, n * 0.5), max(8, n * 0.4))

    fig, ax = plt.subplots(figsize=figsize)

    # Use short labels
    labels = [VAR_LABELS.get(v, v)[:20] for v in corr_df.columns]

    mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)

    sns.heatmap(corr_df, mask=mask, annot=n <= 40, fmt=".2f" if n <= 30 else ".2f",
                cmap="RdBu_r", center=0, vmin=-1, vmax=1,
                square=True, linewidths=0.5,
                xticklabels=labels, yticklabels=labels,
                cbar_kws={"shrink": 0.8, "label": "Correlation"},
                annot_kws={"size": 7} if n <= 30 else {"size": 7},
                ax=ax)

    ax.set_title(title, fontsize=14, fontweight="bold", pad=15)
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"  Heatmap saved: {output_path}")


# ============================================================
# Main
# ============================================================
def main():
    # ---- Read selected features ----
    if not os.path.exists(SELECTED_FEATURES_PATH):
        print(f"Error: {SELECTED_FEATURES_PATH} not found"); sys.exit(1)

    df_sel = pd.read_csv(SELECTED_FEATURES_PATH)
    col_map = {c.lower().strip(): c for c in df_sel.columns}
    var_col = None
    for k in ["variable", "feature", "var", "name"]:
        if k in col_map: var_col = col_map[k]; break
    if var_col is None: var_col = df_sel.columns[0]
    selected = df_sel[var_col].astype(str).str.strip().tolist()
    selected = [v for v in selected if v not in {OUTCOME_VAR, INDEX_VAR, DATE_VAR, "Dataset"}]
    print(f"Selected features: {len(selected)}")

    # ---- Read training data ----
    if not os.path.exists(TRAIN_PATH):
        print(f"Error: {TRAIN_PATH} not found"); sys.exit(1)

    df = pd.read_csv(TRAIN_PATH)
    print(f"Training data: {len(df):,} rows, {len(df.columns)} columns")

    # Validate
    selected = [v for v in selected if v in df.columns]
    print(f"Features in data: {len(selected)}")

    sel_cont = [v for v in CONTINUOUS_VARS if v in selected]
    sel_cat = [v for v in selected if v not in sel_cont]
    print(f"  Continuous: {len(sel_cont)}, Categorical/Binary: {len(sel_cat)}")

    # ---- Prepare numeric data ----
    print("\nPreparing numeric encoding...")
    df_num = prepare_numeric(df, selected)
    print(f"  Numeric matrix: {df_num.shape}")

    # ---- Compute correlation matrices ----
    print("\nComputing Spearman correlation (all features)...")
    corr_all = df_num[selected].corr(method="spearman")

    # P-values for all pairs
    n_feat = len(selected)
    p_matrix = pd.DataFrame(np.ones((n_feat, n_feat)), index=selected, columns=selected)
    for i in range(n_feat):
        for j in range(i + 1, n_feat):
            x = df_num[selected[i]]; y = df_num[selected[j]]
            valid = x.notna() & y.notna()
            if valid.sum() > 3:
                try:
                    _, p = stats.spearmanr(x[valid], y[valid])
                    p_matrix.iloc[i, j] = p; p_matrix.iloc[j, i] = p
                except:
                    pass
    print(f"  Matrix size: {n_feat} x {n_feat}")

    # Continuous only
    corr_cont = None
    p_cont = None
    if len(sel_cont) >= 2:
        print("Computing Spearman correlation (continuous only)...")
        corr_cont = df_num[sel_cont].corr(method="spearman")
        nc = len(sel_cont)
        p_cont = pd.DataFrame(np.ones((nc, nc)), index=sel_cont, columns=sel_cont)
        for i in range(nc):
            for j in range(i + 1, nc):
                x = df_num[sel_cont[i]]; y = df_num[sel_cont[j]]
                valid = x.notna() & y.notna()
                if valid.sum() > 3:
                    try:
                        _, p = stats.spearmanr(x[valid], y[valid])
                        p_cont.iloc[i, j] = p; p_cont.iloc[j, i] = p
                    except:
                        pass

    # ---- Generate heatmaps ----
    print("\nGenerating heatmaps...")
    plot_heatmap(corr_all, OUTPUT_HEATMAP_ALL,
                 f"Spearman Correlation - All Selected Features (n={n_feat})")
    if corr_cont is not None and len(sel_cont) >= 2:
        plot_heatmap(corr_cont, OUTPUT_HEATMAP_CONT,
                     f"Spearman Correlation - Continuous Features (n={len(sel_cont)})")

    # ---- High correlation pairs ----
    all_pairs = []
    for i in range(n_feat):
        for j in range(i + 1, n_feat):
            r_val = corr_all.iloc[i, j]
            p_val = p_matrix.iloc[i, j]
            if pd.notna(r_val):
                all_pairs.append((selected[i], selected[j], r_val, p_val, abs(r_val)))
    all_pairs.sort(key=lambda x: x[4], reverse=True)

    high = [p for p in all_pairs if p[4] >= HIGH_CORR_THRESHOLD]
    print(f"\nHigh correlation pairs (|r| >= {HIGH_CORR_THRESHOLD}): {len(high)}")
    for v1, v2, rv, _, av in high[:10]:
        print(f"  {v1} <-> {v2}: r = {rv:.3f}")
    if len(high) > 10:
        print(f"  ... and {len(high) - 10} more")

    # Save high correlation pairs CSV
    if all_pairs:
        df_pairs = pd.DataFrame(all_pairs, columns=["Variable1", "Variable2", "r", "p_value", "abs_r"])
        df_pairs.to_csv(OUTPUT_HIGH_CORR, index=False)
        print(f"  Saved: {OUTPUT_HIGH_CORR}")

    # ---- VIF ----
    vif_max = None
    if len(sel_cont) >= 2:
        print("\nComputing VIF...")
        try:
            from statsmodels.stats.outliers_influence import variance_inflation_factor
            X = df_num[sel_cont].dropna()
            X_std = (X - X.mean()) / X.std()
            X_std = X_std.dropna(axis=1)
            vifs = []
            for i in range(len(X_std.columns)):
                try:
                    vifs.append(variance_inflation_factor(X_std.values, i))
                except:
                    vifs.append(np.nan)
            vif_max = np.nanmax(vifs) if vifs else None
            n_high_vif = sum(1 for v in vifs if not np.isnan(v) and v >= VIF_THRESHOLD)
            print(f"  Max VIF: {vif_max:.2f}, Features with VIF >= {VIF_THRESHOLD}: {n_high_vif}")
        except ImportError:
            print("  Warning: statsmodels not available, skipping VIF")
            vif_max = None

    # ---- Generate Excel ----
    print("\nGenerating Excel report...")
    wb = Workbook()
    ws_default = wb.active; ws_default.title = "Correlation Summary"

    # Sheet 1: Summary (placeholder, fill after all data ready)
    # Sheet 2: Correlation Matrix (All)
    print("  Sheet: Correlation Matrix (All)")
    write_corr_matrix(wb, corr_all, "Corr Matrix (All)", p_matrix)

    # Sheet 3: Correlation Matrix (Continuous)
    if corr_cont is not None:
        print("  Sheet: Correlation Matrix (Continuous)")
        write_corr_matrix(wb, corr_cont, "Corr Matrix (Continuous)", p_cont)

    # Sheet 4: High Correlation Pairs
    print("  Sheet: High Correlation Pairs")
    write_high_corr_pairs(wb, corr_all, p_matrix, HIGH_CORR_THRESHOLD)

    # Sheet 5: VIF
    print("  Sheet: VIF Analysis")
    write_vif(wb, df_num, sel_cont)

    # Sheet 6: Feature-Outcome Correlation
    if OUTCOME_VAR in df.columns:
        print("  Sheet: Feature-Outcome Correlation")
        write_outcome_corr(wb, df, df_num, selected)

    # Now fill Sheet 1 (Summary)
    # Remove default and recreate at the right position
    wb.remove(ws_default)
    write_summary(wb, corr_all, all_pairs, n_feat, len(sel_cont), vif_max)
    # Move Summary to first position
    summary_ws = wb["Correlation Summary"]
    wb.move_sheet(summary_ws, offset=-(len(wb.sheetnames) - 1))

    wb.save(OUTPUT_XLSX)
    print(f"\nReport saved: {OUTPUT_XLSX}")

    print(f"\n{'='*80}")
    print("CORRELATION ANALYSIS COMPLETE")
    print(f"{'='*80}")
    print(f"  Excel:    {OUTPUT_XLSX}")
    print(f"  Heatmap:  {OUTPUT_HEATMAP_ALL}")
    if corr_cont is not None:
        print(f"  Heatmap:  {OUTPUT_HEATMAP_CONT}")
    print(f"  CSV:      {OUTPUT_HIGH_CORR}")
    print(f"\n  Features analyzed: {n_feat}")
    print(f"  High correlation pairs (|r|>={HIGH_CORR_THRESHOLD}): {len(high)}")
    if vif_max is not None:
        print(f"  Max VIF: {vif_max:.2f}")


if __name__ == "__main__":
    main()

Selected features: 36
Training data: 360,363 rows, 39 columns
Features in data: 36
  Continuous: 22, Categorical/Binary: 14

Preparing numeric encoding...
  Numeric matrix: (360363, 36)

Computing Spearman correlation (all features)...
  Matrix size: 36 x 36
Computing Spearman correlation (continuous only)...

Generating heatmaps...
  Heatmap saved: ./Data\correlation_heatmap.png
  Heatmap saved: ./Data\correlation_heatmap_continuous.png

High correlation pairs (|r| >= 0.7): 6
  LeftTestVol <-> RightTestVol: r = 0.950
  NeutrophilPct <-> LymphocytePct: r = -0.937
  HusbandOcc <-> WifeOcc: r = 0.787
  WifeAge <-> HusbandAge: r = 0.768
  HusbandEdu <-> WifeEdu: r = 0.751
  WifeStress <-> WifeFinance: r = 0.709
  Saved: ./Data\high_correlation_pairs.csv

Computing VIF...
  Max VIF: 11.80, Features with VIF >= 10.0: 2

Generating Excel report...
  Sheet: Correlation Matrix (All)
  Sheet: Correlation Matrix (Continuous)
  Sheet: High Correlation Pairs
  Sheet: VIF Analysis
  Sheet: Feature-